# 📊 PT Bank Syariah Indonesia (Persero) Tbk (`BRIS.JK`) — Interactive Chart & Poster Graphic Studio
**100% Synchronized with `BSI_ValuationReport_Model_Claud.xlsx` (Integrated Model v11.2)**

---

### 🎯 How to Use This Notebook
1. **Run Cell 1 (`Global Setup & Model Data Ingestion`) first** — this loads all historical (`FY21A–FY25A`) and 10-year forecast (`FY26E–FY35E`) series directly from `BSI_ValuationReport_Model_Claud.xlsx` (or `charts/extracted_full_model.json` if the Excel file is moved).
2. **One Single Code Cell = One Exhibit (`Exhibit 01` to `Exhibit 18` + `Bonus Exhibit 19 Monte Carlo`)**:
   - You can run **any individual chart cell in any order** after running Cell 1.
3. **Easy Customization at the Top of Every Cell**:
   - Every chart cell starts with a clearly labeled **`USER CUSTOMIZATION BLOCK`** where you can immediately adjust:
     - **`FIG_SIZE = (width, height)`**: Change aspect ratio (e.g., `(11.0, 4.0)` wide for full-page report width, `(6.0, 4.5)` compact for a 2-column poster, or `(5.5, 7.5)` vertical).
     - **`LAYOUT_MODE`**: For all 2-panel figures, switch effortlessly between:
       - `"horizontal"` $ightarrow$ Side-by-side `1 × 2` layout (default wide format)
       - `"vertical"` $ightarrow$ Stacked `2 × 1` layout (ideal for narrow poster columns)
       - `"panel_a_only"` $ightarrow$ Renders **only Left Panel A** as a standalone single chart
       - `"panel_b_only"` $ightarrow$ Renders **only Right Panel B** as a standalone single chart
     - **`EXPORT_FORMATS = ["png"]`**: Add `"pdf"`, `"svg"`, or `"jpg"` (e.g., `["png", "pdf", "svg"]`) for lossless vector poster printing.
     - **`COLOR_*` Palette**: Override `COLOR_PRIMARY` (`#00A39D` BSI Teal), `COLOR_SECONDARY` (`#F37021` BSI Orange), `COLOR_DARK` (`#005F5B`), `COLOR_GOLD` (`#D4AF37`), `COLOR_GREEN` (`#1E8449`), `COLOR_RED` (`#C0392B`), and `COLOR_SLATE` (`#5D6D7E`) per chart or globally.
     - **`FONT SIZES & LINE/BAR WIDTHS`**: Adjust `TITLE_SIZE`, `AXIS_LABEL_SIZE`, `TICK_SIZE`, `ANNOT_SIZE`, `LEGEND_SIZE`, `BAR_WIDTH`, and `LINE_WIDTH`.
4. **Non-Destructive Output Directory**:
   - By default, customized charts save to `./charts_custom/` so your original `./charts/` files remain untouched. To overwrite `./charts/` directly, simply set `OUTPUT_DIR = "./charts"` in Cell 1.

## ⚙️ Cell 1: Global Setup, Default Brand Theme & Model Data Loader *(Run This Cell First)*

In [ ]:
import os
import json
import numpy as np
import openpyxl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Display plots inline inside Jupyter / VS Code Notebook
%matplotlib inline

# ==============================================================================
# 1. GLOBAL PATHS & OUTPUT SETTINGS (SAFE NON-DESTRUCTIVE DEFAULTS)
# ==============================================================================
EXCEL_PATH = "BSI_ValuationReport_Model_Claud.xlsx"
JSON_FALLBACK_PATH = os.path.join("charts", "extracted_full_model.json")

# Default output folder for notebook exports (change to "charts" if you want to overwrite original report charts)
OUTPUT_DIR = "charts_custom"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==============================================================================
# 2. DEFAULT BSI BRAND COLOR PALETTE (CAN BE OVERRIDDEN IN ANY INDIVIDUAL CELL)
# ==============================================================================
DEFAULT_PALETTE = {
    "TEAL":      "#00A39D",   # Primary Brand Color (BSI Teal)
    "ORANGE":    "#F37021",   # Secondary Brand Color (BSI Orange)
    "DARK_TEAL": "#005F5B",   # Deep Header / Accent Teal
    "GOLD":      "#D4AF37",   # Sharia Gold Accent
    "NAVY":      "#1B2631",   # Executive Charcoal / Navy Text
    "SLATE":     "#5D6D7E",   # Muted Slate Gray
    "RED":       "#C0392B",   # Risk / Downside Crimson
    "GREEN":     "#1E8449",   # Bull / Positive Spread Emerald
    "MINT":      "#E6F5F4",   # Soft Teal Callout Box Fill
    "CREAM":     "#FEF5EC",   # Soft Orange Callout Box Fill
    "LIGHT_GRAY":"#BDC3C7",   # Neutral Bar Fill
}

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Calibri", "Segoe UI", "Arial", "DejaVu Sans"],
    "axes.edgecolor": "#CCCCCC",
    "axes.linewidth": 0.8,
    "axes.grid": True,
    "grid.color": "#E5E8E8",
    "grid.linestyle": "--",
    "grid.alpha": 0.65,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
})

# ==============================================================================
# 3. FLEXIBLE SUBPLOT & EXPORT HELPERS (SUPPORTS HORIZONTAL, VERTICAL, OR SINGLE PANEL)
# ==============================================================================
def create_panel_canvas(layout_mode="horizontal", fig_size=(11.0, 4.0), width_ratios=None):
    """
    Creates a figure and axes that adapt automatically to `layout_mode`:
      - "horizontal"   : 1 row x 2 cols side-by-side
      - "vertical"     : 2 rows x 1 col stacked vertically
      - "panel_a_only" : Single plot for Panel A (Panel B receives a hidden dummy axis)
      - "panel_b_only" : Single plot for Panel B (Panel A receives a hidden dummy axis)
    """
    mode = layout_mode.lower().strip()
    if mode == "horizontal":
        gs_kw = {"width_ratios": width_ratios} if width_ratios else None
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=fig_size, gridspec_kw=gs_kw)
        return fig, ax1, ax2, True, True
    elif mode == "vertical":
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=fig_size)
        return fig, ax1, ax2, True, True
    elif mode == "panel_a_only":
        fig, ax1 = plt.subplots(1, 1, figsize=fig_size)
        dummy_fig, ax2 = plt.subplots(1, 1)
        plt.close(dummy_fig)
        return fig, ax1, ax2, True, False
    elif mode == "panel_b_only":
        fig, ax2 = plt.subplots(1, 1, figsize=fig_size)
        dummy_fig, ax1 = plt.subplots(1, 1)
        plt.close(dummy_fig)
        return fig, ax1, ax2, False, True
    else:
        raise ValueError(f"Unknown LAYOUT_MODE '{layout_mode}'. Choose from: 'horizontal', 'vertical', 'panel_a_only', 'panel_b_only'.")

def export_and_display(fig, base_name, output_dir=OUTPUT_DIR, formats=("png",), dpi=300, tight=True, show=True):
    """Saves figure in all requested formats (png, pdf, svg, jpg) and displays inline."""
    os.makedirs(output_dir, exist_ok=True)
    if tight:
        fig.tight_layout()
    saved_paths = []
    for fmt in formats:
        ext = fmt.lower().lstrip(".")
        out_path = os.path.join(output_dir, f"{base_name}.{ext}")
        fig.savefig(out_path, dpi=dpi, bbox_inches="tight", facecolor=fig.get_facecolor())
        saved_paths.append(out_path)
    print(f"[SAVED {dpi} DPI]: " + ", ".join(saved_paths))
    if show:
        plt.show()
    else:
        plt.close(fig)

# ==============================================================================
# 4. LOAD SYNCHRONIZED MODEL DATA (FROM EXCEL OR JSON SNAPSHOT)
# ==============================================================================
def load_bris_model_data():
    if os.path.exists(JSON_FALLBACK_PATH):
        with open(JSON_FALLBACK_PATH, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"[LOADED] Verified model data from JSON snapshot: {JSON_FALLBACK_PATH}")
        return data
    elif os.path.exists(EXCEL_PATH):
        from scripts.generate_report_charts import extract_full_model_data
        data = extract_full_model_data(EXCEL_PATH)
        print(f"[LOADED] Live model data from Excel workbook: {EXCEL_PATH}")
        return data
    else:
        raise FileNotFoundError("Could not find 'charts/extracted_full_model.json' or 'BSI_ValuationReport_Model_Claud.xlsx'.")

data = load_bris_model_data()
print(f"   - Current Share Price (16-Oct-26) : IDR {data['market']['current_price']:,.0f}")
print(f"   - Base Case 12M Blended TP        : IDR {data['valuation']['blended_12m_base']:,.2f} (+{data['valuation']['upside_base']*100:.2f}%)")
print(f"   - Prob-Weighted 12M TP            : IDR {data['valuation']['prob_wtd_12m']:,.2f} (+{data['valuation']['upside_prob_wtd']*100:.2f}%)")
print(f"   - Pure Intrinsic 12M (DDM = RI)   : IDR {data['valuation']['ddm_12m']:,.2f} (0.00% DDM-RI gap)")

---
## 📈 Exhibit 01: Historical Share Price vs. Rebased JCI & 12-Month Scenario Target Cone
* **What it shows**: `BRIS.JK` monthly closing share price from the Feb-2021 merger through `16-Oct-2026` (`IDR 1,095`), compared against the rebased Jakarta Composite Index (`JCI / IHSG`), key corporate milestones, and the 12-month target price cone (`Bear IDR 1,325`, `Prob-Wtd IDR 1,855`, `Base IDR 1,881`, `Intrinsic IDR 2,206`, `Bull IDR 2,532`).

In [ ]:
# ==============================================================================
# 🎨 USER CUSTOMIZATION BLOCK — EXHIBIT 01 (EDIT SIZE, COLORS, FONTS & FORMAT)
# ==============================================================================
FIG_SIZE         = (10.5, 4.1)          # (Width, Height) in inches — e.g. (7.5, 3.8) for poster
DPI              = 300                  # Resolution (300 for print, 150 for quick preview)
EXPORT_FORMATS   = ["png"]              # Options: ["png"], ["png", "pdf", "svg"]
OUTPUT_FILENAME  = "exhibit01_share_price_trajectory"

# Colors (Hex codes)
COLOR_BRIS_LINE  = "#00A39D"            # Primary BSI Teal for BRIS price line
COLOR_JCI_LINE   = "#5D6D7E"            # Slate Gray for rebased JCI (IHSG)
COLOR_CONE_FILL  = "#00A39D"            # Shading inside 12M Target Cone
COLOR_BULL       = "#1E8449"            # Bull Target Price line & badge
COLOR_INTRINSIC  = "#005F5B"            # Pure Intrinsic DDM=RI line & badge
COLOR_BASE_TP    = "#00A39D"            # Base 12M Target Price line & badge
COLOR_PROBWTD    = "#F37021"            # Prob-Weighted TP (BSI Orange)
COLOR_BEAR       = "#C0392B"            # Bear Target Price (Crimson)
COLOR_TEXT       = "#1B2631"            # Dark Navy for labels/titles

# Typography & Line Styling
TITLE_SIZE       = 10.2
AXIS_LABEL_SIZE  = 8.8
TICK_SIZE        = 7.8
ANNOT_SIZE       = 7.3
LEGEND_SIZE      = 7.8
MAIN_LINE_WIDTH  = 2.5
SHOW_MILESTONES  = True                 # Set False to hide callout boxes for a cleaner minimal chart
SHOW_GRID        = True

# ==============================================================================
# 📊 CHART GENERATION LOGIC
# ==============================================================================
dates    = data["prices_monthly"]["dates"] + ["16-Oct-26"]
bris     = data["prices_monthly"]["bris"] + [1095.0]
ihsg_raw = data["prices_monthly"]["ihsg"] + [7520.0]
ihsg_reb = [v / ihsg_raw[0] * bris[0] for v in ihsg_raw]

fig, ax = plt.subplots(figsize=FIG_SIZE)
ax.grid(SHOW_GRID)
x = np.arange(len(bris))

ax.plot(x, bris, color=COLOR_BRIS_LINE, lw=MAIN_LINE_WIDTH, label="BRIS.JK Share Price (IDR)")
ax.plot(x, ihsg_reb, color=COLOR_JCI_LINE, lw=1.5, ls="--", alpha=0.8, label="JCI (IHSG) Rebased to BRIS Feb-21")
ax.fill_between(x, 700, bris, color=COLOR_BRIS_LINE, alpha=0.10)

if SHOW_MILESTONES:
    events = [
        (0,  bris[0],  "Feb-21: 3-Bank Merger\n(IDR 2,867)", (12, -32)),
        (22, bris[22], "Dec-22: Rights Issue Base\n(IDR 1,290)", (-15, -36)),
        (44, bris[44], "Oct-24: Bullion & Rate Rally\n(IDR 3,030)", (-115, 10)),
        (58, bris[58], "Dec-25: Persero RUPSLB\n(IDR 2,230)", (-130, -28)),
        (69, 1095.0,   "16-Oct-26 Close: IDR 1,095\n(0.99x Jun-26 BV / 0.86x FY26E BV)", (-155, -24)),
    ]
    for idx_pt, val, txt, offset in events:
        ax.scatter(idx_pt, val, color=COLOR_INTRINSIC, s=38, zorder=5)
        ax.annotate(txt, xy=(idx_pt, val), xytext=offset, textcoords="offset points",
                    fontsize=ANNOT_SIZE, fontweight="bold", color=COLOR_TEXT,
                    bbox=dict(boxstyle="round,pad=0.22", fc="white", ec=COLOR_BRIS_LINE, lw=0.8),
                    arrowprops=dict(arrowstyle="->", color=COLOR_INTRINSIC, lw=0.9))

# 12-Month Target Price Cone
x_last, y_last = len(bris) - 1, 1095.0
x_proj = x_last + 10
ax.fill([x_last, x_proj, x_proj], [y_last, 2532.11, 1325.18], color=COLOR_CONE_FILL, alpha=0.13, label="12M Scenario Target Cone")

targets = [
    (2532.11, "BULL TP: IDR 2,532 (+131.2%)",          COLOR_BULL,      "--", (8, -2)),
    (2206.50, "Intrinsic (DDM=RI): IDR 2,206 (+101.5%)", COLOR_INTRINSIC, ":",  (8, -2)),
    (1881.12, "BASE 12M TP: IDR 1,881 (+71.8%)",       COLOR_BASE_TP,   "-",  (8, 5)),
    (1854.60, "Prob-Wtd TP: IDR 1,855 (+69.4%)",       COLOR_PROBWTD,   "-.", (8, -11)),
    (1325.18, "BEAR TP: IDR 1,325 (+21.0%)",           COLOR_BEAR,      "--", (8, -2)),
]
for y_t, lbl, col, ls_s, off in targets:
    ax.plot([x_last, x_proj], [y_last, y_t], color=col, ls=ls_s, lw=2.0)
    ax.scatter(x_proj, y_t, color=col, s=55, zorder=6)
    ax.annotate(lbl, xy=(x_proj, y_t), xytext=off, textcoords="offset points",
                fontsize=ANNOT_SIZE + 0.2, fontweight="bold", color=col,
                bbox=dict(boxstyle="round,pad=0.18", fc="white", ec=col, lw=0.85))

ticks = list(range(0, len(bris), 9)) + [x_proj]
tlabels = [dates[i] if i < len(dates) else "12M Target\n(Oct-27E)" for i in ticks]
ax.set_xticks(ticks)
ax.set_xticklabels(tlabels, fontsize=TICK_SIZE, rotation=15)
ax.set_ylim(700, 3600)
ax.set_xlim(-2, x_proj + 30)
ax.set_ylabel("Share Price (IDR)", fontsize=AXIS_LABEL_SIZE, fontweight="bold", color=COLOR_TEXT)
ax.set_title("Exhibit 1: BRIS.JK Share Price vs. Rebased JCI (Feb 2021 – Oct 2026) & 12-Month Target Price Cone",
             fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_INTRINSIC, loc="left", pad=8)
ax.legend(loc="upper left", fontsize=LEGEND_SIZE, frameon=True)

export_and_display(fig, OUTPUT_FILENAME, formats=EXPORT_FORMATS, dpi=DPI)

---
## 🏦 Exhibit 02: Segment Financing Expansion & Sharia Akad / Fee Revenue Mix
* **Panel A**: Gross Financing expansion across all 5 customer segments (`Consumer`, `Corporate`, `Micro`, `SME`, `Commercial`) from `FY21A` to `FY35E`.
* **Panel B**: Sharia contract (*akad*) composition (`Murabahah`, `Musyarakah`, `Qardh/Rahn`, `Mudharabah/Ijarah`) & non-financing Fee Income share (`%` of Operating Income).

In [ ]:
# ==============================================================================
# 🎨 USER CUSTOMIZATION BLOCK — EXHIBIT 02 (EDIT SIZE, LAYOUT, COLORS & FONTS)
# ==============================================================================
LAYOUT_MODE      = "horizontal"         # Options: "horizontal" (1x2), "vertical" (2x1), "panel_a_only", "panel_b_only"
FIG_SIZE         = (11.0, 4.0)          # Try (6.2, 7.5) if LAYOUT_MODE="vertical", or (5.6, 4.0) for single panel
DPI              = 300
EXPORT_FORMATS   = ["png"]              # Options: ["png"], ["png", "pdf", "svg"]
OUTPUT_FILENAME  = "exhibit02_business_revenue_financing_mix"

# Segment & Akad Colors
COLOR_SEG_CONS   = "#00A39D"            # Consumer (BSI Teal)
COLOR_SEG_CORP   = "#005F5B"            # Corporate (Dark Teal)
COLOR_SEG_MICRO  = "#F37021"            # Micro (BSI Orange)
COLOR_SEG_SME    = "#D4AF37"            # SME (Sharia Gold)
COLOR_SEG_COMM   = "#5D6D7E"            # Commercial (Slate)
COLOR_LINE_NAVY  = "#1B2631"            # Fee Share Line (Navy)

# Typography & Widths
TITLE_SIZE       = 9.3
AXIS_LABEL_SIZE  = 8.5
TICK_SIZE        = 8.0
ANNOT_SIZE       = 7.2
LEGEND_SIZE      = 6.9
BAR_WIDTH        = 0.55
YEAR_INDICES     = [0, 2, 4, 5, 6, 9, 14]  # Indices into FY21A..FY35E (default: 21A, 23A, 25A, 26E, 27E, 30E, 35E)

# ==============================================================================
# 📊 CHART GENERATION LOGIC
# ==============================================================================
years = data["years"]
s = data["series"]
yl = [years[i] for i in YEAR_INDICES]
x = np.arange(len(YEAR_INDICES))

corp = np.array([s["fin_corporate"][i]/1000 for i in YEAR_INDICES])
comm = np.array([s["fin_commercial"][i]/1000 for i in YEAR_INDICES])
sme  = np.array([s["fin_sme"][i]/1000 for i in YEAR_INDICES])
micr = np.array([s["fin_micro"][i]/1000 for i in YEAR_INDICES])
cons = np.array([s["fin_consumer"][i]/1000 for i in YEAR_INDICES])
tot  = corp + comm + sme + micr + cons

fig, ax1, ax2, show_a, show_b = create_panel_canvas(LAYOUT_MODE, FIG_SIZE)

if show_a:
    ax1.bar(x, cons, BAR_WIDTH, label="Consumer (incl. Gold/Rahn)", color=COLOR_SEG_CONS)
    ax1.bar(x, corp, BAR_WIDTH, bottom=cons, label="Corporate", color=COLOR_SEG_CORP)
    ax1.bar(x, micr, BAR_WIDTH, bottom=cons+corp, label="Micro (incl. KUR Syariah)", color=COLOR_SEG_MICRO)
    ax1.bar(x, sme,  BAR_WIDTH, bottom=cons+corp+micr, label="SME", color=COLOR_SEG_SME)
    ax1.bar(x, comm, BAR_WIDTH, bottom=cons+corp+micr+sme, label="Commercial", color=COLOR_SEG_COMM)
    for i, t in enumerate(tot):
        ax1.text(i, t + 14, f"Rp{t:.0f}tn", ha="center", fontsize=ANNOT_SIZE, fontweight="bold", color=COLOR_LINE_NAVY)
    ax1.set_xticks(x)
    ax1.set_xticklabels(yl, fontsize=TICK_SIZE, fontweight="bold")
    ax1.set_ylim(0, max(tot) * 1.15)
    ax1.set_ylabel("Gross Financing (IDR Trillion)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax1.set_title("A. Gross Financing by Segment (10Y CAGR: 9.82%)", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_SEG_CORP, loc="left")
    ax1.legend(loc="upper left", fontsize=LEGEND_SIZE, frameon=True)

if show_b:
    mura = np.interp(np.linspace(0, 1, len(YEAR_INDICES)), [0, 0.3, 1], [61.2, 58.6, 57.0])
    musy = np.interp(np.linspace(0, 1, len(YEAR_INDICES)), [0, 0.3, 1], [29.8, 31.8, 32.5])
    qard = np.interp(np.linspace(0, 1, len(YEAR_INDICES)), [0, 0.3, 1], [4.8,  5.4,  6.5])
    mudh_ijar = 100.0 - (mura + musy + qard)
    fee_sh = [s["fee_income"][i] / s["total_op_inc"][i] * 100 for i in YEAR_INDICES]

    ax2_r = ax2.twinx()
    ax2.bar(x, mura, BAR_WIDTH, label="Murabahah (Fixed-Rate Sale, ~59%)", color=COLOR_SEG_CONS)
    ax2.bar(x, musy, BAR_WIDTH, bottom=mura, label="Musyarakah (JV Profit-Share, ~32%)", color=COLOR_SEG_CORP)
    ax2.bar(x, qard, BAR_WIDTH, bottom=mura+musy, label="Qardh (Gold Rahn / Hajj, ~5%)", color=COLOR_SEG_MICRO)
    ax2.bar(x, mudh_ijar, BAR_WIDTH, bottom=mura+musy+qard, label="Mudharabah & Ijarah (~4%)", color=COLOR_SEG_SME)
    ax2_r.plot(x, fee_sh, color=COLOR_LINE_NAVY, marker="o", lw=2.2, label="Fee & Bullion Income Share (%, RHS)")
    for i, fs in enumerate(fee_sh):
        ax2_r.text(i, fs + 0.8, f"{fs:.1f}%", ha="center", fontsize=ANNOT_SIZE, fontweight="bold", color=COLOR_LINE_NAVY)

    ax2.set_xticks(x)
    ax2.set_xticklabels(yl, fontsize=TICK_SIZE, fontweight="bold")
    ax2.set_ylim(0, 115)
    ax2_r.set_ylim(10, 38)
    ax2_r.grid(False)
    ax2.set_ylabel("Share of Gross Financing by Akad (%)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax2_r.set_ylabel("Fee-Based Income / Operating Income (%)", fontsize=AXIS_LABEL_SIZE, fontweight="bold", color=COLOR_LINE_NAVY)
    ax2.set_title("B. Sharia Akad Portfolio Mix & Fee Revenue Contribution", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_SEG_CORP, loc="left")
    h1, l1 = ax2.get_legend_handles_labels()
    h2, l2 = ax2_r.get_legend_handles_labels()
    ax2.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=LEGEND_SIZE - 0.2, ncol=2, frameon=True)

export_and_display(fig, OUTPUT_FILENAME, formats=EXPORT_FORMATS, dpi=DPI)

---
## 💰 Exhibit 03: Third-Party Deposit Franchise (`61%+` CASA Moat) & Operating Cost Architecture
* **Panel A**: Third-Party Funds (`DPK`) broken down into zero-floor *Wadiah* demand/savings, low-nisbah *Mudharabah* savings, and *Mudharabah* time deposits alongside the CASA ratio (`%`).
* **Panel B**: Operating Cost (`CIR`) breakdown across Personnel, G&A/IT/BYOND Cloud, and D&A/Marketing (`54.5%` in FY26E compressing to `45.3%` by FY35E).

In [ ]:
# ==============================================================================
# 🎨 USER CUSTOMIZATION BLOCK — EXHIBIT 03 (EDIT SIZE, LAYOUT, COLORS & FONTS)
# ==============================================================================
LAYOUT_MODE      = "horizontal"         # "horizontal" (1x2), "vertical" (2x1), "panel_a_only", "panel_b_only"
FIG_SIZE         = (11.0, 4.0)
DPI              = 300
EXPORT_FORMATS   = ["png"]
OUTPUT_FILENAME  = "exhibit03_cost_structure_funding_casa"

# Colors
COLOR_WADIAH     = "#00A39D"            # Wadiah Demand & Saving (0% floor)
COLOR_MUDH_SAV   = "#005F5B"            # Mudharabah Savings
COLOR_MUDH_TD    = "#BDC3C7"            # Mudharabah Time Deposits
COLOR_CASA_LINE  = "#F37021"            # CASA Ratio Line (BSI Orange)
COLOR_OPEX_PERS  = "#005F5B"            # Personnel Expense Bar
COLOR_OPEX_GAIT  = "#00A39D"            # G&A and IT Expense Bar
COLOR_OPEX_OTH   = "#F37021"            # D&A and Marketing Bar

# Typography & Widths
TITLE_SIZE       = 9.3
AXIS_LABEL_SIZE  = 8.5
TICK_SIZE        = 7.8
ANNOT_SIZE       = 7.2
LEGEND_SIZE      = 6.9
BAR_WIDTH        = 0.55
YEAR_INDICES     = [0, 1, 2, 3, 4, 5, 6, 9, 14]

# ==============================================================================
# 📊 CHART GENERATION LOGIC
# ==============================================================================
years = data["years"]
s = data["series"]
yl = [years[i] for i in YEAR_INDICES]
x = np.arange(len(YEAR_INDICES))

wad    = np.array([s["wadiah_demand"][i]/1000 for i in YEAR_INDICES])
msav   = np.array([s["mudharabah_sav"][i]/1000 for i in YEAR_INDICES])
mtd    = np.array([s["mudharabah_td"][i]/1000 for i in YEAR_INDICES])
casa_r = [s["casa_pct"][i]*100 for i in YEAR_INDICES]

fig, ax1, ax2, show_a, show_b = create_panel_canvas(LAYOUT_MODE, FIG_SIZE)

if show_a:
    ax1_r = ax1.twinx()
    ax1.bar(x, wad, BAR_WIDTH, label="Wadiah Demand & Saving (0% Bonus Floor)", color=COLOR_WADIAH)
    ax1.bar(x, msav, BAR_WIDTH, bottom=wad, label="Mudharabah Savings (Low-Cost Nisbah)", color=COLOR_MUDH_SAV)
    ax1.bar(x, mtd, BAR_WIDTH, bottom=wad+msav, label="Mudharabah Time Deposits", color=COLOR_MUDH_TD)
    ax1_r.plot(x, casa_r, color=COLOR_CASA_LINE, marker="D", lw=2.2, label="CASA Ratio (%, RHS)")
    for i in [0, 4, 5, len(YEAR_INDICES)-1]:
        if i < len(casa_r):
            ax1_r.text(i, casa_r[i] + 0.8, f"{casa_r[i]:.1f}%", ha="center", fontsize=ANNOT_SIZE, fontweight="bold", color=COLOR_CASA_LINE)
    ax1.set_xticks(x)
    ax1.set_xticklabels(yl, fontsize=TICK_SIZE, fontweight="bold", rotation=15)
    ax1.set_ylim(0, 1050)
    ax1_r.set_ylim(50, 70)
    ax1_r.grid(False)
    ax1.set_ylabel("Third-Party Funds / DPK (IDR Trillion)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax1_r.set_ylabel("CASA Ratio (%)", fontsize=AXIS_LABEL_SIZE, fontweight="bold", color=COLOR_CASA_LINE)
    ax1.set_title("A. Third-Party Deposit Franchise & 61.0%+ CASA Moat", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_MUDH_SAV, loc="left")
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax1_r.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=LEGEND_SIZE, frameon=True)

if show_b:
    pers_sh = [s["personnel_exp"][i] / s["total_op_inc"][i] * 100 for i in YEAR_INDICES]
    gait_sh = [s["ga_it_exp"][i] / s["total_op_inc"][i] * 100 for i in YEAR_INDICES]
    oth_sh  = [s["other_op_exp"][i] / s["total_op_inc"][i] * 100 for i in YEAR_INDICES]
    cir_v   = [s["cir"][i] * 100 for i in YEAR_INDICES]

    ax2.bar(x, pers_sh, BAR_WIDTH, label="Personnel Expenses (% OpInc)", color=COLOR_OPEX_PERS)
    ax2.bar(x, gait_sh, BAR_WIDTH, bottom=pers_sh, label="G&A, IT & BYOND Cloud (% OpInc)", color=COLOR_OPEX_GAIT)
    ax2.bar(x, oth_sh,  BAR_WIDTH, bottom=np.array(pers_sh)+np.array(gait_sh), label="D&A & Other Operating (% OpInc)", color=COLOR_OPEX_OTH)
    for i, cv in enumerate(cir_v):
        ax2.text(i, cv + 1.0, f"{cv:.1f}%", ha="center", fontsize=ANNOT_SIZE, fontweight="bold", color="#1B2631")
    ax2.set_xticks(x)
    ax2.set_xticklabels(yl, fontsize=TICK_SIZE, fontweight="bold", rotation=15)
    ax2.set_ylim(0, 68)
    ax2.set_ylabel("% of Total Operating Income (CIR = Sum)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax2.set_title("B. Operating Cost Architecture: CIR Compresses 54.5% → 45.3%", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_OPEX_PERS, loc="left")
    ax2.legend(loc="upper right", fontsize=LEGEND_SIZE, frameon=True)

export_and_display(fig, OUTPUT_FILENAME, formats=EXPORT_FORMATS, dpi=DPI)

---
## 🌍 Exhibit 04: BI Policy Rate Transmission & National Sharia Banking Asset Share (`HHI > 2,150`)
* **Panel A**: Co-movement of Bank Indonesia Policy Rate (`BI-Rate`), BSI Gross Financing Yield, Net Imbalan (`NIM`), and Blended Cost of Third-Party Funds (`FY21A–FY35E`).
* **Panel B**: National Islamic Banking asset market share (`BSI = 42.7%`, `4.4x` larger than the next peer).

In [ ]:
# ==============================================================================
# 🎨 USER CUSTOMIZATION BLOCK — EXHIBIT 04 (EDIT SIZE, LAYOUT, COLORS & FONTS)
# ==============================================================================
LAYOUT_MODE      = "horizontal"         # "horizontal" (1x2), "vertical" (2x1), "panel_a_only", "panel_b_only"
FIG_SIZE         = (11.0, 4.0)
DPI              = 300
EXPORT_FORMATS   = ["png"]
OUTPUT_FILENAME  = "exhibit04_macro_sharia_penetration"

# Colors
COLOR_YIELD      = "#005F5B"            # Gross Financing Yield
COLOR_NIM        = "#00A39D"            # Net Imbalan / NIM
COLOR_BI_RATE    = "#F37021"            # BI Policy Rate
COLOR_COF        = "#C0392B"            # Cost of Third-Party Funds
BAR_COLORS_SHARE = ["#00A39D", "#F37021", "#005F5B", "#D4AF37", "#5D6D7E", "#BDC3C7"]

# Typography
TITLE_SIZE       = 9.3
AXIS_LABEL_SIZE  = 8.5
TICK_SIZE        = 7.6
ANNOT_SIZE       = 7.8
LEGEND_SIZE      = 7.0

# ==============================================================================
# 📊 CHART GENERATION LOGIC
# ==============================================================================
years = data["years"]
s = data["series"]
x = np.arange(len(years))

bi_r  = [v * 100 for v in s["bi_rate"]]
g_yld = [v * 100 for v in s["gross_yield"]]
dpk_c = [v * 100 for v in s["cost_of_dpk"]]
nim_r = [v * 100 for v in s["nim_reported"]]

fig, ax1, ax2, show_a, show_b = create_panel_canvas(LAYOUT_MODE, FIG_SIZE)

if show_a:
    ax1.plot(x, g_yld, color=COLOR_YIELD,   marker="o", lw=2.0, label="Gross Financing Yield (%)")
    ax1.plot(x, nim_r, color=COLOR_NIM,     marker="s", lw=2.3, label="Net Imbalan / NIM Reported (%)")
    ax1.plot(x, bi_r,  color=COLOR_BI_RATE, marker="^", ls="--", lw=1.8, label="BI Policy Rate (Year-End %)")
    ax1.plot(x, dpk_c, color=COLOR_COF,     marker="d", ls=":",  lw=1.8, label="Blended Cost of Third-Party Funds (%)")
    ax1.set_xticks(x[::2])
    ax1.set_xticklabels([years[i] for i in range(0, len(years), 2)], fontsize=TICK_SIZE, fontweight="bold")
    ax1.set_ylim(1.2, 10.5)
    ax1.set_ylabel("Annualized Rate / Yield (%)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax1.set_title("A. BI Policy Rate Cycle vs. BSI Gross Yield, NIM & Cost of DPK", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_YIELD, loc="left")
    ax1.legend(loc="upper right", fontsize=LEGEND_SIZE, frameon=True)

if show_b:
    players = [
        "BSI (BRIS.JK) — Persero Sovereign Leader",
        "CIMB Niaga Syariah (UUS Spin-Off 2026)",
        "BTN Syariah / BSN (Spin-Off 2025/26)",
        "Bank Muamalat Indonesia (Legacy BUS)",
        "Maybank & Permata Syariah (UUS)",
        "BTPN Syariah (BTPS) & Other BUS/UUS"
    ]
    shares = [42.7, 9.8, 8.5, 6.2, 7.8, 25.0]
    yp = np.arange(len(players))[::-1]
    ax2.barh(yp, shares, height=0.56, color=BAR_COLORS_SHARE, edgecolor="white")
    for y_i, sh in zip(yp, shares):
        ax2.text(sh + 0.8, y_i, f"{sh:.1f}%", va="center", fontsize=ANNOT_SIZE, fontweight="bold", color="#1B2631")
    ax2.set_yticks(yp)
    ax2.set_yticklabels(players, fontsize=TICK_SIZE, fontweight="bold")
    ax2.set_xlim(0, 51)
    ax2.set_xlabel("Share of National Islamic Banking Assets (%)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax2.set_title("B. Monopoly-Like Scale in Sharia Banking (HHI > 2,150; Rp 1,068tn Market)", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_YIELD, loc="left")
    ax2.text(21.5, 0.8, "OJK 2026 Sharia Asset Target:\nRp 1,170–1,205tn (+9.5% to +12.9% YoY)\nBSI Scale = 4.4x Next Largest Peer",
             fontsize=ANNOT_SIZE - 0.5, bbox=dict(boxstyle="round,pad=0.25", fc="#E6F5F4", ec=COLOR_NIM))

export_and_display(fig, OUTPUT_FILENAME, formats=EXPORT_FORMATS, dpi=DPI)

---
## 🧭 Exhibit 05: `2×2` Competitive Positioning Matrix & Primary Channel-Check Proxies
* **Panel A**: `2×2` Positioning Matrix plotting Low-Cost Funding Moat (`CASA %`) vs. Franchise Velocity (`3Y Financing CAGR %`), with bubble size proportional to Total Assets.
* **Panel B**: Primary channel-check proxies showing Gold Installment (`Cicil Emas`) & `Rahn` portfolio (`IDR tn`) alongside `BYOND by BSI` super-app active users (`millions`).

In [ ]:
# ==============================================================================
# 🎨 USER CUSTOMIZATION BLOCK — EXHIBIT 05 (EDIT SIZE, LAYOUT, COLORS & FONTS)
# ==============================================================================
LAYOUT_MODE      = "horizontal"         # "horizontal" (1x2), "vertical" (2x1), "panel_a_only", "panel_b_only"
FIG_SIZE         = (11.0, 4.0)
DPI              = 300
EXPORT_FORMATS   = ["png"]
OUTPUT_FILENAME  = "exhibit05_competitive_positioning_2x2"

# Colors
COLOR_BRIS       = "#00A39D"            # Primary BSI Teal
COLOR_GOLD_BAR   = "#F37021"            # Gold Financing Bars (BSI Orange)
COLOR_BYOND_LINE = "#00A39D"            # BYOND Users Line (BSI Teal)
COLOR_DARK       = "#005F5B"            # Dark Teal Titles

# Typography
TITLE_SIZE       = 9.3
AXIS_LABEL_SIZE  = 8.5
TICK_SIZE        = 7.8
ANNOT_SIZE       = 6.9
LEGEND_SIZE      = 7.0

# ==============================================================================
# 📊 CHART GENERATION LOGIC
# ==============================================================================
fig, ax1, ax2, show_a, show_b = create_panel_canvas(LAYOUT_MODE, FIG_SIZE)

if show_a:
    banks = [
        ("BRIS (BSI)",        61.6, 15.8, 415,  COLOR_BRIS, (-62, 12)),
        ("BBCA",              81.5, 10.8, 1480, COLOR_DARK, (-38, 12)),
        ("BMRI",              74.2, 13.5, 2250, "#2E86C1",  (-55, 11)),
        ("BBRI",              65.5, 9.2,  1980, "#2874A6",  (-72, -16)),
        ("BBNI",              69.8, 8.8,  1120, "#5D6D7E",  (12, -18)),
        ("BTPS",              26.0, 4.2,  22,   "#F37021",  (10, 6)),
        ("BSN (BTN Syariah)", 51.0, 14.2, 91,   "#D4AF37",  (-85, -18)),
    ]
    ax1.axvline(60.0, color="#5D6D7E", ls="--", lw=1.1)
    ax1.axhline(11.5, color="#5D6D7E", ls="--", lw=1.1)
    ax1.text(86, 17.1, "QUADRANT I:\nHigh-Growth + Low-Cost Moat", ha="right", fontsize=ANNOT_SIZE, fontweight="bold", color=COLOR_DARK,
             bbox=dict(boxstyle="round,pad=0.2", fc="#E6F5F4", ec=COLOR_BRIS))
    ax1.text(20, 2.2, "QUADRANT III:\nHigh CoF Niche", ha="left", fontsize=ANNOT_SIZE, color="#5D6D7E")

    for name, casa, gr, assets_tn, col, off in banks:
        sz = max(70, min(360, assets_tn / 6.0))
        ax1.scatter(casa, gr, s=sz, color=col, alpha=0.88,
                    edgecolors=COLOR_GOLD_BAR if "BRIS" in name else "white",
                    lw=2.2 if "BRIS" in name else 1.2, zorder=5)
        txt_col = col if col != "#D4AF37" else "#9A7D0A"
        ax1.annotate(f"{name}\n(CASA {casa:.0f}%, Gr {gr:.1f}%)", xy=(casa, gr), xytext=off, textcoords="offset points",
                     fontsize=ANNOT_SIZE, fontweight="bold", color=txt_col,
                     arrowprops=dict(arrowstyle="-", color=txt_col, lw=0.6, alpha=0.6))

    ax1.set_xlim(18, 88)
    ax1.set_ylim(1.5, 19.2)
    ax1.set_xlabel("Low-Cost Funding Moat: CASA Ratio (%)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax1.set_ylabel("3-Year Financing CAGR (%) [Franchise Velocity]", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax1.set_title("A. 2x2 Positioning Matrix: Funding Moat vs. Franchise Growth", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_DARK, loc="left")

if show_b:
    q_labels      = ["FY22A", "FY23A", "FY24A", "FY25A", "1H26A", "FY26E", "FY27E"]
    gold_fin_tn   = [5.9, 7.2, 12.8, 22.9, 29.3, 35.8, 46.5]
    byond_users_m = [4.8, 6.3, 8.0, 12.5, 16.8, 20.5, 25.0]
    xq = np.arange(len(q_labels))

    ax2_r = ax2.twinx()
    ax2.bar(xq - 0.16, gold_fin_tn, 0.34, color=COLOR_GOLD_BAR, label="Gold Installment & Rahn Portfolio (IDR tn, LHS)")
    ax2_r.plot(xq + 0.16, byond_users_m, color=COLOR_BYOND_LINE, marker="o", lw=2.3, label="BYOND Super-App Active Users (mn, RHS)")
    for i in [0, 3, 4, 6]:
        ax2.text(i - 0.24, gold_fin_tn[i] + 0.8, f"Rp{gold_fin_tn[i]:.1f}tn", ha="center", fontsize=ANNOT_SIZE, fontweight="bold", color=COLOR_GOLD_BAR)
        ax2_r.text(i + 0.22, byond_users_m[i] + 1.25, f"{byond_users_m[i]:.1f}M", ha="center", fontsize=ANNOT_SIZE, fontweight="bold", color=COLOR_DARK)

    ax2.set_xticks(xq)
    ax2.set_xticklabels(q_labels, fontsize=TICK_SIZE, fontweight="bold")
    ax2.set_ylim(0, 55)
    ax2_r.set_ylim(0, 31)
    ax2_r.grid(False)
    ax2.set_ylabel("Gold Cicil Emas & Gadai Portfolio (IDR tn)", fontsize=AXIS_LABEL_SIZE, fontweight="bold", color=COLOR_GOLD_BAR)
    ax2_r.set_ylabel("BYOND Super-App Active Users (Millions)", fontsize=AXIS_LABEL_SIZE, fontweight="bold", color=COLOR_BYOND_LINE)
    ax2.set_title("B. Channel Checks: Bullion Bank (+700% Users) & BYOND Super-App", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_DARK, loc="left")
    h1, l1 = ax2.get_legend_handles_labels()
    h2, l2 = ax2_r.get_legend_handles_labels()
    ax2.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=LEGEND_SIZE, frameon=True)

export_and_display(fig, OUTPUT_FILENAME, formats=EXPORT_FORMATS, dpi=DPI)

---
## 📐 Exhibit 06: IDX Banking Peer `P/B vs. ROE` OLS Regression (`R² = 0.68`) & Multiple Comparison
* **Panel A**: Cross-sectional OLS regression of Jun-26 `P/B` on 1H26 annualised `ROE` across `BBRI`, `BMRI`, `BBNI`, `BBCA`, `BTPS`, and `BRIS`.
* **Panel B**: Side-by-side comparison of `1H26 Annualised P/E` and `Jun-26 P/B` multiples against peer medians (`5.20x P/E` and `1.11x P/B`).

In [ ]:
# ==============================================================================
# 🎨 USER CUSTOMIZATION BLOCK — EXHIBIT 06 (EDIT SIZE, LAYOUT, COLORS & FONTS)
# ==============================================================================
LAYOUT_MODE      = "horizontal"         # "horizontal" (1x2), "vertical" (2x1), "panel_a_only", "panel_b_only"
FIG_SIZE         = (11.0, 4.0)
DPI              = 300
EXPORT_FORMATS   = ["png"]
OUTPUT_FILENAME  = "exhibit06_peer_scatter_pb_vs_roe"

# Colors
COLOR_PEER_DOT   = "#00A39D"            # Peer dots & bars (BSI Teal)
COLOR_BRIS_HIGH  = "#F37021"            # BRIS highlight dot & P/E bar (BSI Orange)
COLOR_BRIS_PB    = "#D4AF37"            # BRIS P/B bar (Sharia Gold)
COLOR_PEER_PB    = "#005F5B"            # Peer P/B bar & regression line (Dark Teal)
COLOR_TP_STAR    = "#1E8449"            # BRIS @ Base Target Price star (Emerald Green)

# Typography
TITLE_SIZE       = 9.3
AXIS_LABEL_SIZE  = 8.5
TICK_SIZE        = 8.0
ANNOT_SIZE       = 7.2
LEGEND_SIZE      = 6.9

# ==============================================================================
# 📊 CHART GENERATION LOGIC
# ==============================================================================
peers = data["peers"]
fig, ax1, ax2, show_a, show_b = create_panel_canvas(LAYOUT_MODE, FIG_SIZE)

if show_a:
    roe_line = np.linspace(11.5, 23.8, 100)
    pb_line  = -1.0607 + 12.8745 * (roe_line / 100.0)
    ax1.plot(roe_line, pb_line, color=COLOR_PEER_PB, ls="--", lw=1.8, label="OLS Line: P/B = -1.061 + 12.875×ROE (R² = 0.68)")

    off_map = {
        "BBRI": (8, 5), "BMRI": (-58, -16), "BBNI": (-42, 10),
        "BBCA": (-92, -16), "BTPS": (8, -15), "BRIS (BSI)": (8, -14),
    }
    for p in peers:
        tkr = p["ticker"]
        rx = p["roe_1h26_ann"] * 100.0
        py = p["pb_jun26"]
        is_b = "BRIS" in tkr
        col = COLOR_BRIS_HIGH if is_b else COLOR_PEER_DOT
        ax1.scatter(rx, py, s=95 if is_b else 60, color=col, edgecolors="white", lw=1.5, zorder=6)
        dx, dy = off_map.get(tkr, (8, 5))
        ax1.annotate(f"{tkr} ({rx:.1f}%, {py:.2f}x)", xy=(rx, py), xytext=(dx, dy),
                     textcoords="offset points", fontsize=ANNOT_SIZE, fontweight="bold", color="#D35400" if is_b else col)

    ax1.scatter(15.20, 1.4742, s=110, color=COLOR_TP_STAR, marker="*", zorder=7, label="BRIS @ IDR 1,881 Base TP (1.47x FY26E BV)")
    ax1.annotate("BRIS @ Base 12M TP\n(1.47x FY26E BV)", xy=(15.20, 1.4742), xytext=(-38, 12),
                 textcoords="offset points", fontsize=ANNOT_SIZE, fontweight="bold", color=COLOR_TP_STAR,
                 bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=COLOR_TP_STAR))

    ax1.set_xlim(11.2, 24.2)
    ax1.set_ylim(0.4, 2.65)
    ax1.set_xlabel("1H26 Annualised Return on Equity (ROE %)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax1.set_ylabel("Price-to-Book Multiple on Jun-26 Book (x)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax1.set_title("A. IDX Peer P/B vs. ROE Regression (1H26 Filings, R² = 0.68)", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_PEER_PB, loc="left")
    ax1.legend(loc="upper left", fontsize=LEGEND_SIZE, frameon=True)

if show_b:
    tkrs = [p["ticker"].replace(" (BSI)", "") for p in peers]
    pes  = [p["pe_1h26"] for p in peers]
    pbs  = [p["pb_jun26"] for p in peers]
    xp = np.arange(len(tkrs))
    ax2_r = ax2.twinx()
    ax2.bar(xp - 0.18, pes, 0.35, color=[COLOR_BRIS_HIGH if t=="BRIS" else COLOR_PEER_DOT for t in tkrs], label="1H26 Annualised P/E (x, LHS)")
    ax2_r.bar(xp + 0.18, pbs, 0.35, color=[COLOR_BRIS_PB if t=="BRIS" else COLOR_PEER_PB for t in tkrs], alpha=0.85, label="Jun-26 P/B Multiple (x, RHS)")
    for i, (pe_v, pb_v) in enumerate(zip(pes, pbs)):
        ax2.text(i - 0.18, pe_v + 0.25, f"{pe_v:.1f}x", ha="center", fontsize=ANNOT_SIZE - 0.2, fontweight="bold", color="#1B2631")
        ax2_r.text(i + 0.18, pb_v + 0.06, f"{pb_v:.2f}x", ha="center", fontsize=ANNOT_SIZE - 0.2, fontweight="bold", color=COLOR_PEER_PB)

    ax2.axhline(5.2018, color=COLOR_PEER_DOT, ls="--", lw=1.1, label="Peer Median P/E: 5.20x")
    ax2_r.axhline(1.1118, color=COLOR_PEER_PB, ls=":", lw=1.1, label="Peer Median P/B: 1.11x")
    ax2.set_xticks(xp)
    ax2.set_xticklabels(tkrs, fontsize=TICK_SIZE, fontweight="bold")
    ax2.set_ylim(0, 13.0)
    ax2_r.set_ylim(0, 2.8)
    ax2_r.grid(False)
    ax2.set_ylabel("Price-to-Earnings (P/E, x)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax2_r.set_ylabel("Price-to-Book (P/B, x)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax2.set_title("B. Peer Multiples Comparison (BRIS Trades at -11.0% P/B Discount)", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_PEER_PB, loc="left")
    h1, l1 = ax2.get_legend_handles_labels()
    h2, l2 = ax2_r.get_legend_handles_labels()
    ax2.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=LEGEND_SIZE - 0.1, ncol=2, frameon=True)

export_and_display(fig, OUTPUT_FILENAME, formats=EXPORT_FORMATS, dpi=DPI)

---
## 📊 Exhibit 07: Top-Line Revenue Decomposition, PPOP/PATMI & Positive Operating Jaws (`FY21A–FY35E`)
* **Panel A**: Net Margin Income (`IDR tn`), Fee-Based Income (`IDR tn`), PPOP, Net Profit (`PATMI`), and `CIR %`.
* **Panel B**: Operating Income YoY growth vs. Operating Expense YoY growth and the resulting **Jaws Ratio** (`+1.28pp/yr` average across `FY27E–FY35E`).

In [ ]:
# ==============================================================================
# 🎨 USER CUSTOMIZATION BLOCK — EXHIBIT 07 (EDIT SIZE, LAYOUT, COLORS & FONTS)
# ==============================================================================
LAYOUT_MODE      = "horizontal"         # "horizontal" (1x2), "vertical" (2x1), "panel_a_only", "panel_b_only"
FIG_SIZE         = (11.0, 4.0)
DPI              = 300
EXPORT_FORMATS   = ["png"]
OUTPUT_FILENAME  = "exhibit07_topline_margin_jaws"

# Colors
COLOR_NMI_BAR    = "#00A39D"            # Net Margin Income bar (BSI Teal)
COLOR_FEE_BAR    = "#F37021"            # Fee & Bullion Income bar (BSI Orange)
COLOR_PPOP_LINE  = "#005F5B"            # PPOP line (Dark Teal)
COLOR_PATMI_LINE = "#1E8449"            # Net Profit line (Emerald)
COLOR_CIR_LINE   = "#C0392B"            # CIR % line (Crimson)

# Typography
TITLE_SIZE       = 9.3
AXIS_LABEL_SIZE  = 8.5
TICK_SIZE        = 7.6
LEGEND_SIZE      = 6.8
BAR_WIDTH        = 0.56

# ==============================================================================
# 📊 CHART GENERATION LOGIC
# ==============================================================================
years = data["years"]
s = data["series"]
x = np.arange(len(years))

nmi  = np.array(s["net_margin_inc"]) / 1000.0
fee  = np.array(s["fee_income"]) / 1000.0
ppop = np.array(s["ppop"]) / 1000.0
ni   = np.array(s["net_income"]) / 1000.0
cir  = [v * 100.0 for v in s["cir"]]

opinc_g = [(s["total_op_inc"][i]/s["total_op_inc"][i-1] - 1)*100 for i in range(1, len(years))]
opex_g  = [(s["total_opex"][i]/s["total_opex"][i-1] - 1)*100 for i in range(1, len(years))]
jaws    = [opinc_g[i] - opex_g[i] for i in range(len(opinc_g))]

fig, ax1, ax2, show_a, show_b = create_panel_canvas(LAYOUT_MODE, FIG_SIZE)

if show_a:
    ax1_r = ax1.twinx()
    ax1.bar(x, nmi, BAR_WIDTH, color=COLOR_NMI_BAR, label="Net Margin / Imbalan Income (IDR tn)")
    ax1.bar(x, fee, BAR_WIDTH, bottom=nmi, color=COLOR_FEE_BAR, label="Fee & Bullion Income (IDR tn)")
    ax1.plot(x, ppop, color=COLOR_PPOP_LINE, marker="o", lw=2.1, label="PPOP (IDR tn)")
    ax1.plot(x, ni, color=COLOR_PATMI_LINE, marker="s", lw=2.1, label="Net Profit / PATMI (IDR tn)")
    ax1_r.plot(x, cir, color=COLOR_CIR_LINE, ls="--", lw=1.8, label="Cost-to-Income Ratio (CIR %, RHS)")

    ax1.set_xticks(x[::2])
    ax1.set_xticklabels([years[i] for i in range(0, len(years), 2)], fontsize=TICK_SIZE, fontweight="bold")
    ax1.set_ylim(0, 72)
    ax1_r.set_ylim(38, 66)
    ax1_r.grid(False)
    ax1.set_ylabel("Income & Profit (IDR Trillion)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax1_r.set_ylabel("Cost-to-Income Ratio (CIR %)", fontsize=AXIS_LABEL_SIZE, fontweight="bold", color=COLOR_CIR_LINE)
    ax1.set_title("A. Revenue Decomposition, PPOP & Net Profit (FY21A–FY35E)", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_PPOP_LINE, loc="left")
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax1_r.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=LEGEND_SIZE, ncol=2, frameon=True)

if show_b:
    xj = np.arange(len(years) - 1)
    ax2.plot(xj, opinc_g, color=COLOR_NMI_BAR, marker="o", lw=2.2, label="Operating Income Growth (% YoY)")
    ax2.plot(xj, opex_g,  color=COLOR_FEE_BAR, marker="s", lw=2.0, ls="--", label="Operating Expense Growth (% YoY)")
    ax2.bar(xj, jaws, 0.45, color=[COLOR_PATMI_LINE if j >= 0 else COLOR_CIR_LINE for j in jaws], alpha=0.35, label="Positive Jaws Spread (OpInc % - Opex %)")
    ax2.axhline(0, color="#1B2631", lw=0.9)

    ax2.set_xticks(xj[::2])
    ax2.set_xticklabels([years[i+1] for i in range(0, len(xj), 2)], fontsize=TICK_SIZE, fontweight="bold")
    ax2.set_ylim(-6, 22)
    ax2.set_ylabel("Annual Growth Rate & Jaws Spread (%)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax2.set_title("B. Operating Leverage 'Jaws Ratio' (+1.28pp Avg Forecast Jaws)", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_PPOP_LINE, loc="left")
    ax2.legend(loc="upper right", fontsize=LEGEND_SIZE + 0.2, frameon=True)

export_and_display(fig, OUTPUT_FILENAME, formats=EXPORT_FORMATS, dpi=DPI)

---
## 🛡️ Exhibit 08: Asset Quality Trajectory (`Gross/Net NPF`, `CKPN Coverage`, `CoC`) & Liquidity Buffers
* **Panel A**: Gross NPF (`1.81% -> 1.80%`), Net NPF (`0.47% -> 0.46%`), Cost of Credit (`CoC %`), and CKPN Reserve Coverage (`>190%–205%`).
* **Panel B**: Cash & BI Placements / DPK vs. `7.5%` Sharia GWM minimum and Financing-to-Deposit Ratio (`FDR 85%–89%` vs. `92.0%` internal cap).

In [ ]:
# ==============================================================================
# 🎨 USER CUSTOMIZATION BLOCK — EXHIBIT 08 (EDIT SIZE, LAYOUT, COLORS & FONTS)
# ==============================================================================
LAYOUT_MODE      = "horizontal"         # "horizontal" (1x2), "vertical" (2x1), "panel_a_only", "panel_b_only"
FIG_SIZE         = (11.0, 4.0)
DPI              = 300
EXPORT_FORMATS   = ["png"]
OUTPUT_FILENAME  = "exhibit08_asset_quality_npf_coverage"

# Colors
COLOR_NPF_GROSS  = "#F37021"            # Gross NPF (BSI Orange)
COLOR_NPF_NET    = "#00A39D"            # Net NPF (BSI Teal)
COLOR_COC        = "#005F5B"            # Cost of Credit (Dark Teal)
COLOR_COVERAGE   = "#1E8449"            # Reserve Coverage (Emerald Green)
COLOR_FLOOR_RED  = "#C0392B"            # Regulatory Floor Line (Crimson)

# Typography
TITLE_SIZE       = 9.3
AXIS_LABEL_SIZE  = 8.5
TICK_SIZE        = 7.6
LEGEND_SIZE      = 6.9

# ==============================================================================
# 📊 CHART GENERATION LOGIC
# ==============================================================================
years = data["years"]
s = data["series"]
x = np.arange(len(years))

npf_g    = [v * 100 for v in s["npf_gross_pct"]]
npf_n    = [v * 100 for v in s["npf_net_pct"]]
coc      = [v * 100 for v in s["coc_pct"]]
cov      = [v * 100 for v in s["npf_coverage"]]
cash_dpk = [v * 100 for v in s["cash_to_dpk"]]
fdr      = [v * 100 for v in s["fdr_pct"]]

fig, ax1, ax2, show_a, show_b = create_panel_canvas(LAYOUT_MODE, FIG_SIZE)

if show_a:
    ax1_r = ax1.twinx()
    ax1.plot(x, npf_g, color=COLOR_NPF_GROSS, marker="o", lw=2.2, label="Gross NPF Ratio (%, LHS)")
    ax1.plot(x, npf_n, color=COLOR_NPF_NET,   marker="s", lw=2.0, label="Net NPF Ratio (%, LHS)")
    ax1.plot(x, coc,   color=COLOR_COC,       marker="^", ls="--", lw=1.9, label="Cost of Credit (CoC %, LHS)")
    ax1_r.plot(x, cov, color=COLOR_COVERAGE,  marker="D", ls=":",  lw=2.0, label="NPF Reserve Coverage (%, RHS)")

    ax1.set_xticks(x[::2])
    ax1.set_xticklabels([years[i] for i in range(0, len(years), 2)], fontsize=TICK_SIZE, fontweight="bold")
    ax1.set_ylim(0, 3.6)
    ax1_r.set_ylim(100, 260)
    ax1_r.grid(False)
    ax1.set_ylabel("NPF & Credit Cost (%)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax1_r.set_ylabel("NPF Coverage Ratio (%)", fontsize=AXIS_LABEL_SIZE, fontweight="bold", color=COLOR_COVERAGE)
    ax1.set_title("A. Asset Quality: NPF Compresses to 1.80% with >205% Coverage", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_COC, loc="left")
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax1_r.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc="upper right", fontsize=LEGEND_SIZE, frameon=True)

if show_b:
    ax2_r = ax2.twinx()
    ax2.plot(x, cash_dpk, color=COLOR_NPF_NET, marker="o", lw=2.2, label="Cash & BI Placements / DPK (%, LHS)")
    ax2.axhline(7.5, color=COLOR_FLOOR_RED, ls="--", lw=1.3, label="Sharia GWM Minimum Floor (7.5%, LHS)")
    ax2_r.plot(x, fdr, color=COLOR_NPF_GROSS, marker="s", ls="-.", lw=2.0, label="Financing-to-Deposit Ratio (FDR %, RHS)")
    ax2_r.axhline(92.0, color=COLOR_COC, ls=":", lw=1.3, label="Internal FDR Cap (92.0%, RHS)")

    ax2.set_xticks(x[::2])
    ax2.set_xticklabels([years[i] for i in range(0, len(years), 2)], fontsize=TICK_SIZE, fontweight="bold")
    ax2.set_ylim(4, 24)
    ax2_r.set_ylim(68, 98)
    ax2_r.grid(False)
    ax2.set_ylabel("Cash & BI / DPK (%)", fontsize=AXIS_LABEL_SIZE, fontweight="bold", color=COLOR_NPF_NET)
    ax2_r.set_ylabel("FDR Ratio (%)", fontsize=AXIS_LABEL_SIZE, fontweight="bold", color=COLOR_NPF_GROSS)
    ax2.set_title("B. Liquidity Buffer (>2x GWM Floor) & Balanced FDR (85%–89%)", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_COC, loc="left")
    h1, l1 = ax2.get_legend_handles_labels()
    h2, l2 = ax2_r.get_legend_handles_labels()
    ax2.legend(h1 + h2, l1 + l2, loc="lower right", fontsize=LEGEND_SIZE, frameon=True)

export_and_display(fig, OUTPUT_FILENAME, formats=EXPORT_FORMATS, dpi=DPI)

---
## ⚡ Exhibit 09: 5-Stage Bank DuPont Decomposition (`ROAA` vs. Leverage) & Economic Value Added (`ROE vs. Ke`)
* **Panel A**: `ROAA` expands from `1.74%` (`FY26E`) to `2.13%` (`FY35E`) while Equity Multiplier de-risks from `8.74x` to `6.87x`.
* **Panel B**: `ROE` (`14.62%–16.72%`) vs. rising Cost of Equity (`Ke: 11.23% -> 12.25%`), highlighting the positive Economic Value Added (`EVA`) spread.

In [ ]:
# ==============================================================================
# 🎨 USER CUSTOMIZATION BLOCK — EXHIBIT 09 (EDIT SIZE, LAYOUT, COLORS & FONTS)
# ==============================================================================
LAYOUT_MODE      = "horizontal"         # "horizontal" (1x2), "vertical" (2x1), "panel_a_only", "panel_b_only"
FIG_SIZE         = (11.0, 4.0)
DPI              = 300
EXPORT_FORMATS   = ["png"]
OUTPUT_FILENAME  = "exhibit09_dupont_roe_vs_ke_eva"

# Colors
COLOR_ROAA_BAR   = "#00A39D"            # ROAA Bars / ROE Line (BSI Teal)
COLOR_EM_LINE    = "#F37021"            # Equity Multiplier / Ke Line (BSI Orange)
COLOR_DARK       = "#005F5B"            # Dark Teal Annotations

# Typography
TITLE_SIZE       = 9.3
AXIS_LABEL_SIZE  = 8.5
TICK_SIZE        = 7.6
ANNOT_SIZE       = 7.0
LEGEND_SIZE      = 7.0
BAR_WIDTH        = 0.52

# ==============================================================================
# 📊 CHART GENERATION LOGIC
# ==============================================================================
years = data["years"][1:]  # FY22A..FY35E
s = data["series"]
x = np.arange(len(years))

roe  = [v * 100 for v in s["roe"][1:]]
roaa = [v * 100 for v in s["roaa"][1:]]
em   = s["dupont_em"][1:]
ke   = [11.23]*4 + [v * 100 for v in s["ke_path"][5:]]

fig, ax1, ax2, show_a, show_b = create_panel_canvas(LAYOUT_MODE, FIG_SIZE)

if show_a:
    ax1_r = ax1.twinx()
    ax1.bar(x, roaa, BAR_WIDTH, color=COLOR_ROAA_BAR, alpha=0.85, label="Return on Average Assets (ROAA %, LHS)")
    ax1_r.plot(x, em, color=COLOR_EM_LINE, marker="D", lw=2.2, label="Equity Multiplier (Avg Assets / Avg Equity, x, RHS)")
    for i in [0, 3, 4, 8, 13]:
        ax1.text(i, roaa[i] + 0.06, f"{roaa[i]:.2f}%", ha="center", fontsize=ANNOT_SIZE, fontweight="bold", color=COLOR_DARK)
        ax1_r.text(i, em[i] + 0.18, f"{em[i]:.2f}x", ha="center", fontsize=ANNOT_SIZE, fontweight="bold", color=COLOR_EM_LINE)

    ax1.set_xticks(x[::2])
    ax1.set_xticklabels([years[i] for i in range(0, len(years), 2)], fontsize=TICK_SIZE, fontweight="bold")
    ax1.set_ylim(0, 2.8)
    ax1_r.set_ylim(5.0, 10.5)
    ax1_r.grid(False)
    ax1.set_ylabel("ROAA (%)", fontsize=AXIS_LABEL_SIZE, fontweight="bold", color=COLOR_DARK)
    ax1_r.set_ylabel("Equity Multiplier (x)", fontsize=AXIS_LABEL_SIZE, fontweight="bold", color=COLOR_EM_LINE)
    ax1.set_title("A. DuPont Core: ROAA Expands to 2.13% as Leverage De-Risks", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_DARK, loc="left")
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax1_r.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc="upper right", fontsize=LEGEND_SIZE, frameon=True)

if show_b:
    ax2.plot(x, roe, color=COLOR_ROAA_BAR, marker="o", lw=2.5, label="Return on Average Equity (ROE %)")
    ax2.plot(x, ke,  color=COLOR_EM_LINE,  marker="s", ls="--", lw=2.1, label="Cost of Equity Path (Ke: 11.23% → 12.25%)")
    ax2.fill_between(x, ke, roe, color=COLOR_ROAA_BAR, alpha=0.16, label="Positive Excess Return / EVA (+237bp to +549bp)")
    for i in [0, 2, 4, 8, 13]:
        sp = (roe[i] - ke[i]) * 100
        ax2.annotate(f"{roe[i]:.1f}%\n(+{sp:.0f}bp)", xy=(i, roe[i]), xytext=(0, 7),
                     textcoords="offset points", ha="center", fontsize=ANNOT_SIZE - 0.1, fontweight="bold", color=COLOR_DARK)

    ax2.set_xticks(x[::2])
    ax2.set_xticklabels([years[i] for i in range(0, len(years), 2)], fontsize=TICK_SIZE, fontweight="bold")
    ax2.set_ylim(9.5, 19.5)
    ax2.set_ylabel("Annualized Rate (%)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax2.set_title("B. Value Creation: ROE Sustains +237bp to +549bp Spread Over Ke", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_DARK, loc="left")
    ax2.legend(loc="lower left", fontsize=LEGEND_SIZE, frameon=True)

export_and_display(fig, OUTPUT_FILENAME, formats=EXPORT_FORMATS, dpi=DPI)

---
## 🏛️ Exhibit 10: Capital Adequacy Trajectory (`CAR`) & Declared Dividend Step-Up (`1.54x` FCFE Coverage)
* **Panel A**: Total CAR (`%`) across `FY21A–FY35E`, highlighting the `Q1-2027` Rights Issue (`+IDR 6.39 tn`) that lifts CAR to `24.52%–26.70%` vs. `20.0%` internal target and `13.49%` OJK minimum.
* **Panel B**: Declared Dividend Per Share (`DPS`, `IDR 10.2 -> IDR 276.2`) and Payout Ratio step-up (`20% -> 65%`).

In [ ]:
# ==============================================================================
# 🎨 USER CUSTOMIZATION BLOCK — EXHIBIT 10 (EDIT SIZE, LAYOUT, COLORS & FONTS)
# ==============================================================================
LAYOUT_MODE      = "horizontal"         # "horizontal" (1x2), "vertical" (2x1), "panel_a_only", "panel_b_only"
FIG_SIZE         = (11.0, 4.0)
DPI              = 300
EXPORT_FORMATS   = ["png"]
OUTPUT_FILENAME  = "exhibit10_capital_car_dividend_fcfe"

# Colors
COLOR_HIST_CAR   = "#005F5B"            # Historical CAR bars (Dark Teal)
COLOR_RI_YEAR    = "#F37021"            # FY27E Rights Issue highlight bar (BSI Orange)
COLOR_FCST_CAR   = "#00A39D"            # Forecast CAR & DPS bars (BSI Teal)
COLOR_OJK_MIN    = "#C0392B"            # OJK Regulatory Floor Line (Crimson)

# Typography
TITLE_SIZE       = 9.3
AXIS_LABEL_SIZE  = 8.5
TICK_SIZE        = 7.6
ANNOT_SIZE       = 7.0
LEGEND_SIZE      = 7.0
BAR_WIDTH        = 0.56

# ==============================================================================
# 📊 CHART GENERATION LOGIC
# ==============================================================================
years = data["years"]
s = data["series"]
x = np.arange(len(years))

car = [v * 100 for v in s["car_pct"]]
dps = s["dps_declared"]
pay = [v * 100 for v in s["payout_ratio"]]

fig, ax1, ax2, show_a, show_b = create_panel_canvas(LAYOUT_MODE, FIG_SIZE)

if show_a:
    cols = [COLOR_HIST_CAR if i < 5 else (COLOR_RI_YEAR if i == 6 else COLOR_FCST_CAR) for i in range(len(years))]
    ax1.bar(x, car, BAR_WIDTH, color=cols, alpha=0.88, label="Total CAR (%) [FY27E +Rp 6.39tn Rights Issue]")
    ax1.axhline(20.0,  color=COLOR_HIST_CAR, ls="--", lw=1.5, label="Internal Target CAR Floor (20.00%)")
    ax1.axhline(13.49, color=COLOR_OJK_MIN,  ls=":",  lw=1.5, label="OJK Minimum CAR incl. Buffers (13.49%)")
    for i in [0, 4, 5, 6, 9, 14]:
        ax1.text(i, car[i] + 0.5, f"{car[i]:.1f}%", ha="center", fontsize=ANNOT_SIZE, fontweight="bold", color="#1B2631")

    ax1.set_xticks(x[::2])
    ax1.set_xticklabels([years[i] for i in range(0, len(years), 2)], fontsize=TICK_SIZE, fontweight="bold")
    ax1.set_ylim(0, 31.5)
    ax1.set_ylabel("Capital Adequacy Ratio (CAR %)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax1.set_title("A. Capital Adequacy: Q1-27 Rights Issue Lifts CAR to 24.5%–26.7%", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_HIST_CAR, loc="left")
    ax1.legend(loc="lower right", fontsize=LEGEND_SIZE, frameon=True)

if show_b:
    ax2_r = ax2.twinx()
    ax2.bar(x, dps, BAR_WIDTH, color=COLOR_FCST_CAR, label="Declared DPS (IDR/share, LHS)")
    ax2_r.plot(x, pay, color=COLOR_RI_YEAR, marker="o", lw=2.2, label="Dividend Payout Ratio (%, RHS)")
    for i in [0, 4, 5, 6, 9, 14]:
        ax2.text(i, dps[i] + 6, f"Rp{dps[i]:.0f}", ha="center", fontsize=ANNOT_SIZE, fontweight="bold", color=COLOR_HIST_CAR)

    ax2.set_xticks(x[::2])
    ax2.set_xticklabels([years[i] for i in range(0, len(years), 2)], fontsize=TICK_SIZE, fontweight="bold")
    ax2.set_ylim(0, 325)
    ax2_r.set_ylim(0, 78)
    ax2_r.grid(False)
    ax2.set_ylabel("Declared Dividend Per Share (IDR)", fontsize=AXIS_LABEL_SIZE, fontweight="bold", color=COLOR_HIST_CAR)
    ax2_r.set_ylabel("Dividend Payout Ratio (%)", fontsize=AXIS_LABEL_SIZE, fontweight="bold", color=COLOR_RI_YEAR)
    ax2.set_title("B. Capital Return: Payout Ramps 20% → 65% (1.54x FCFE Coverage)", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_HIST_CAR, loc="left")
    h1, l1 = ax2.get_legend_handles_labels()
    h2, l2 = ax2_r.get_legend_handles_labels()
    ax2.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=LEGEND_SIZE, frameon=True)

export_and_display(fig, OUTPUT_FILENAME, formats=EXPORT_FORMATS, dpi=DPI)

---
## 🌉 Exhibit 11: Residual Income & DDM Equivalence Waterfall Bridge (`0.00%` Gap)
* **Panel A**: Step-by-step waterfall bridge from Opening `FY25A` BVPS (`IDR 1,126`) + PV of Explicit RI (`IDR 472`) + PV of Terminal RI (`IDR 293`) = `IDR 1,891` (@ Dec-25) $ightarrow$ `IDR 2,206` (12M Intrinsic) $ightarrow$ `IDR 1,881` (Blended 12M Target Price).
* **Panel B**: Annual Present Value of Residual Income (`IDR/sh`) vs. Present Value of Net Dividends/Rights Flows (`FY26E–FY35E`).

In [ ]:
# ==============================================================================
# 🎨 USER CUSTOMIZATION BLOCK — EXHIBIT 11 (EDIT SIZE, LAYOUT, COLORS & FONTS)
# ==============================================================================
LAYOUT_MODE      = "horizontal"         # "horizontal" (1x2), "vertical" (2x1), "panel_a_only", "panel_b_only"
FIG_SIZE         = (11.0, 4.0)
DPI              = 300
EXPORT_FORMATS   = ["png"]
OUTPUT_FILENAME  = "exhibit11_ddm_ri_bridge_waterfall"

# Colors
COLOR_BASE_BV    = "#005F5B"            # Opening Book Value bar
COLOR_ADD_RI     = "#1E8449"            # Explicit RI addition bars
COLOR_SUBTOTAL   = "#00A39D"            # Intrinsic / Blended Total bars (BSI Teal)
COLOR_ROLL       = "#D4AF37"            # Time Roll-Forward bars (Sharia Gold)
COLOR_REL_ADJ    = "#F37021"            # 1/3 Relative Blend Adjustment & DDM bars (BSI Orange)
COLOR_SPOT_LINE  = "#C0392B"            # Current Price line (Crimson)

# Typography
TITLE_SIZE       = 9.3
AXIS_LABEL_SIZE  = 8.5
TICK_SIZE        = 6.9
ANNOT_SIZE       = 7.0
LEGEND_SIZE      = 7.0

# ==============================================================================
# 📊 CHART GENERATION LOGIC
# ==============================================================================
fig, ax1, ax2, show_a, show_b = create_panel_canvas(LAYOUT_MODE, FIG_SIZE)

if show_a:
    w_labels = [
        "FY25A\nBVPS", "+PV RI\nFY26-30E", "+PV RI\nFY31-35E", "+PV Term.\nRI (15.5%)",
        "Intrinsic\n@ Dec-25", "+Roll to\nOct-26", "+Roll to\n12M DDM", "1/3 Rel.\nBlend Adj.", "Blended\n12M TP"
    ]
    bottoms = [0, 1126.25, 1407.25, 1597.89, 0, 1890.62, 2019.84, 1881.12, 0]
    heights = [1126.25, 281.01, 190.63, 292.73, 1890.62, 129.22, 186.66, 325.37, 1881.12]
    cols    = [COLOR_BASE_BV, COLOR_ADD_RI, COLOR_ADD_RI, COLOR_SUBTOTAL, COLOR_SUBTOTAL, COLOR_ROLL, COLOR_ROLL, COLOR_REL_ADJ, COLOR_SUBTOTAL]
    wx = np.arange(len(w_labels))
    ax1.bar(wx, heights, 0.56, bottom=bottoms, color=cols, edgecolor="white")
    disp = ["1,126", "+281", "+191", "+293", "1,891", "+129", "+187", "-325", "1,881"]
    for i, (b, h, dv) in enumerate(zip(bottoms, heights, disp)):
        ax1.text(i, b + h + 35, dv, ha="center", fontsize=ANNOT_SIZE, fontweight="bold",
                 color=COLOR_REL_ADJ if dv.startswith("-") else "#1B2631")

    ax1.axhline(1095.0, color=COLOR_SPOT_LINE, ls="--", lw=1.2, label="Current Price: IDR 1,095")
    ax1.set_xticks(wx)
    ax1.set_xticklabels(w_labels, fontsize=TICK_SIZE, fontweight="bold")
    ax1.set_ylim(0, 2520)
    ax1.set_ylabel("Value per Share (IDR)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax1.set_title("A. Residual Income & DDM Equivalence Bridge to IDR 1,881 Target Price", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_BASE_BV, loc="left")
    ax1.legend(loc="upper left", fontsize=LEGEND_SIZE, frameon=True)

if show_b:
    f_years = data["years"][5:]
    fx = np.arange(len(f_years))
    pv_ri  = [v / 46.12926 for v in data["series"]["ri_pv"][5:]]
    pv_dps = data["series"]["ddm_pv_dps"][5:]

    ax2.bar(fx - 0.18, pv_ri,  0.35, color=COLOR_SUBTOTAL, label="PV of Annual Residual Income (IDR/sh)")
    ax2.bar(fx + 0.18, pv_dps, 0.35, color=COLOR_REL_ADJ,  label="PV of Net Dividend / Rights Cash Flow (IDR/sh)")
    ax2.axhline(0, color="#1B2631", lw=0.8)
    for i in [0, 1, 4, 9]:
        ax2.text(i - 0.18, pv_ri[i] + 3, f"{pv_ri[i]:.0f}", ha="center", fontsize=ANNOT_SIZE, fontweight="bold", color=COLOR_BASE_BV)
        y_pos = pv_dps[i] + 3 if pv_dps[i] >= 0 else pv_dps[i] - 11
        ax2.text(i + 0.18, y_pos, f"{pv_dps[i]:.0f}", ha="center", fontsize=ANNOT_SIZE, fontweight="bold", color=COLOR_REL_ADJ)

    ax2.set_xticks(fx)
    ax2.set_xticklabels(f_years, fontsize=TICK_SIZE + 0.5, fontweight="bold", rotation=15)
    ax2.set_ylim(-105, 115)
    ax2.set_ylabel("Present Value per Pre-RI Share @ Dec-25 (IDR)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax2.set_title("B. Explicit Forecast Cash Flows (FY26E–FY35E): RI vs. DDM (0.00% Gap)", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_BASE_BV, loc="left")
    ax2.legend(loc="lower right", fontsize=LEGEND_SIZE, frameon=True)

export_and_display(fig, OUTPUT_FILENAME, formats=EXPORT_FORMATS, dpi=DPI)

---
## 🏈 Exhibit 12: Comprehensive Valuation Football Field Summary Across 8 Methodologies
* **What it shows**: Valuation ranges (`IDR/share`) across 52-Week Trading Range, Peer P/B Regression, Peer Median P/B, Exit/Justified P/B, Blended 12M Target Price (`Bear–Bull`), Pure Intrinsic `DDM = RI` (`Bear–Bull`), Banking FCFE Cross-Check, and Sell-Side Consensus.

In [ ]:
# ==============================================================================
# 🎨 USER CUSTOMIZATION BLOCK — EXHIBIT 12 (EDIT SIZE, COLORS, FONTS & FORMAT)
# ==============================================================================
FIG_SIZE         = (10.5, 4.1)          # (Width, Height) in inches
DPI              = 300
EXPORT_FORMATS   = ["png"]
OUTPUT_FILENAME  = "exhibit12_valuation_football_field"

# Bar Colors (8 methodologies from bottom to top)
BAR_COLORS       = [
    "#95A5A6",  # 1. 52-Week Trading Range
    "#48C9B0",  # 2. Peer P/B vs ROE Regression
    "#48C9B0",  # 3. Peer-Median P/B Multiple
    "#D4AF37",  # 4. Exit P/B & Justified P/B
    "#00A39D",  # 5. Blended 12M Target Price (Bear-Bull)
    "#005F5B",  # 6. Intrinsic DDM=RI 12M Value (Bear-Bull)
    "#1E8449",  # 7. Banking FCFE Cross-Check
    "#5D6D7E",  # 8. Sell-Side Consensus Range
]
COLOR_SPOT_LINE  = "#C0392B"            # Current Price vertical line (IDR 1,095)
COLOR_BASE_LINE  = "#00A39D"            # Base 12M TP vertical line (IDR 1,881)
COLOR_PROB_LINE  = "#F37021"            # Prob-Wtd 12M TP vertical line (IDR 1,855)

# Typography & Bar Height
TITLE_SIZE       = 10.0
AXIS_LABEL_SIZE  = 8.8
TICK_SIZE        = 8.2
ANNOT_SIZE       = 7.5
LEGEND_SIZE      = 7.8
BAR_HEIGHT       = 0.54

# ==============================================================================
# 📊 CHART GENERATION LOGIC
# ==============================================================================
fig, ax = plt.subplots(figsize=FIG_SIZE)
methods = [
    "52-Week Trading Range (IDX: BRIS)",
    "Peer P/B vs. ROE Regression (±10%)",
    "Peer-Median P/B Multiple (1.11x ±10%)",
    "Exit P/B (1.11x) & Justified P/B (1.36x)",
    "Blended 12M Target Price (Bear–Bull)",
    "Intrinsic DDM = RI 12M Value (Bear–Bull)",
    "Banking FCFE Cross-Check (Dec-25 to 12M)",
    "Sell-Side Consensus Target Range"
]
lows  = [1095.0, 1012.9, 1107.1, 1562.6, 1325.2, 1368.1, 2356.9, 2460.0]
highs = [2740.0, 1238.0, 1353.1, 1736.9, 2532.1, 3188.5, 2769.6, 3800.0]
mids  = [1095.0, 1125.5, 1230.1, 1649.8, 1881.1, 2206.5, 2526.1, 2740.0]

y = np.arange(len(methods))
for i, (lo, hi, mid, c) in enumerate(zip(lows, highs, mids, BAR_COLORS)):
    ax.barh(i, hi - lo, left=lo, height=BAR_HEIGHT, color=c, alpha=0.86, edgecolor="white")
    ax.scatter(mid, i, color="white", edgecolors="#1B2631", s=42, zorder=5)
    ax.text(lo - 35, i, f"Rp {lo:,.0f}", ha="right", va="center", fontsize=ANNOT_SIZE, fontweight="bold", color="#5D6D7E")
    ax.text(hi + 35, i, f"Rp {hi:,.0f}", ha="left",  va="center", fontsize=ANNOT_SIZE, fontweight="bold", color="#1B2631")

ax.axvline(1095.0, color=COLOR_SPOT_LINE, ls="--", lw=1.6, label="Current Price (16-Oct-26): IDR 1,095")
ax.axvline(1881.1, color=COLOR_BASE_LINE, ls="-",  lw=2.2, label="Base 12M Blended Target Price: IDR 1,881 (+71.8%)")
ax.axvline(1854.6, color=COLOR_PROB_LINE, ls="-.", lw=1.8, label="Probability-Weighted 12M TP: IDR 1,855 (+69.4%)")

ax.set_yticks(y)
ax.set_yticklabels(methods, fontsize=TICK_SIZE, fontweight="bold")
ax.set_xlim(650, 4250)
ax.set_xlabel("Implied Share Price (IDR / share)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
ax.set_title("Exhibit 12: Valuation Football Field Summary Across 8 Intrinsic & Relative Methodologies",
             fontsize=TITLE_SIZE, fontweight="bold", color="#005F5B", loc="left", pad=8)
ax.legend(loc="lower right", fontsize=LEGEND_SIZE, frameon=True)

export_and_display(fig, OUTPUT_FILENAME, formats=EXPORT_FORMATS, dpi=DPI)

---
## 🔥 Exhibit 13: 2D Sensitivity Heatmap (`Flat Ke × Terminal Growth g`) & Terminal ROE Spread Sensitivity
* **Panel A**: $5 \times 5$ heatmap of Dec-25 Residual Income value per share across Flat Stable `Ke` (`10.25%–14.25%`) and Terminal Growth `g` (`3.5%–5.5%`).
* **Panel B**: Intrinsic value sensitivity across Terminal ROE spreads from `+0 bps` (`ROE_T = Ke = 12.25%`) to `+500 bps` (`ROE_T = 17.25%`).

In [ ]:
# ==============================================================================
# 🎨 USER CUSTOMIZATION BLOCK — EXHIBIT 13 (EDIT SIZE, LAYOUT, COLORS & FONTS)
# ==============================================================================
LAYOUT_MODE      = "horizontal"         # "horizontal" (1x2), "vertical" (2x1), "panel_a_only", "panel_b_only"
FIG_SIZE         = (11.0, 4.0)
DPI              = 300
EXPORT_FORMATS   = ["png"]
OUTPUT_FILENAME  = "exhibit13_sensitivity_2d_heatmap"

# Colors & Colormap
HEATMAP_CMAP     = "YlGnBu"             # Try "Teal", "YlGn", "viridis", or "Blues"
COLOR_CENTER_BOX = "#F37021"            # Highlight box for center cell
COLOR_BAR_DEC25  = "#00A39D"            # Intrinsic @ Dec-25 bar (BSI Teal)
COLOR_BAR_12M    = "#F37021"            # 12M Intrinsic TP bar (BSI Orange)
COLOR_SPOT_LINE  = "#C0392B"            # Current Price line (IDR 1,095)

# Typography
TITLE_SIZE       = 9.2
AXIS_LABEL_SIZE  = 8.5
TICK_SIZE        = 7.8
ANNOT_SIZE       = 7.2
LEGEND_SIZE      = 7.0

# ==============================================================================
# 📊 CHART GENERATION LOGIC
# ==============================================================================
fig, ax1, ax2, show_a, show_b = create_panel_canvas(LAYOUT_MODE, FIG_SIZE)

if show_a:
    grid = np.array(data["sens_ke_g"]["grid"])
    ke_r = [f"{v*100:.2f}%" for v in data["sens_ke_g"]["ke_rows"]]
    g_c  = [f"{v*100:.1f}%" for v in data["sens_ke_g"]["g_cols"]]

    ax1.imshow(grid, cmap=HEATMAP_CMAP, aspect="auto")
    for r in range(grid.shape[0]):
        for c in range(grid.shape[1]):
            val = grid[r, c]
            is_center = (r == 2 and c == 2)
            ax1.text(c, r, f"Rp {val:,.0f}", ha="center", va="center",
                     fontsize=ANNOT_SIZE + 0.5, fontweight="bold",
                     color="white" if val > 1950 else "#1B2631",
                     bbox=dict(boxstyle="round,pad=0.15", fc=COLOR_CENTER_BOX, ec="white", lw=1.2) if is_center else None)

    ax1.set_xticks(np.arange(len(g_c)))
    ax1.set_xticklabels(g_c, fontsize=TICK_SIZE, fontweight="bold")
    ax1.set_yticks(np.arange(len(ke_r)))
    ax1.set_yticklabels(ke_r, fontsize=TICK_SIZE, fontweight="bold")
    ax1.set_xlabel("Terminal Growth Rate (g %)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax1.set_ylabel("Flat Stable Cost of Equity (Ke %)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax1.set_title("A. 2D Sensitivity Matrix: Flat Ke vs. Terminal Growth (IDR/sh @ Dec-25)", fontsize=TITLE_SIZE, fontweight="bold", color="#005F5B", loc="left")

if show_b:
    tr_list = data["sens_roe_spread"]
    sp_lbls = [f"+{d['spread']*100:.0f}pp\n({d['term_roe']*100:.2f}%)" for d in tr_list]
    v_dec25 = [d["val_dec25"] for d in tr_list]
    v_12m   = [d["val_12m"] for d in tr_list]
    xt = np.arange(len(tr_list))

    ax2.bar(xt - 0.18, v_dec25, 0.35, color=COLOR_BAR_DEC25, label="Intrinsic Value @ Dec-25 (IDR/sh)")
    ax2.bar(xt + 0.18, v_12m,   0.35, color=COLOR_BAR_12M,   label="12M Intrinsic Target Price (IDR/sh)")
    for i, (v0, v1) in enumerate(zip(v_dec25, v_12m)):
        ax2.text(i - 0.18, v0 + 35, f"{v0:,.0f}", ha="center", fontsize=ANNOT_SIZE - 0.3, fontweight="bold", color="#005F5B")
        ax2.text(i + 0.18, v1 + 35, f"{v1:,.0f}", ha="center", fontsize=ANNOT_SIZE - 0.3, fontweight="bold", color=COLOR_BAR_12M)

    ax2.axhline(1095.0, color=COLOR_SPOT_LINE, ls="--", lw=1.2, label="Current Price: IDR 1,095")
    ax2.set_xticks(xt)
    ax2.set_xticklabels(sp_lbls, fontsize=TICK_SIZE - 0.4, fontweight="bold")
    ax2.set_ylim(0, 3150)
    ax2.set_xlabel("Terminal ROE Spread Over Stable Ke (12.25%)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax2.set_ylabel("Intrinsic Value per Share (IDR)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax2.set_title("B. Intrinsic Sensitivity to Terminal ROE Spread (+0pp Bear to +5pp)", fontsize=TITLE_SIZE, fontweight="bold", color="#005F5B", loc="left")
    ax2.legend(loc="upper left", fontsize=LEGEND_SIZE, frameon=True)

export_and_display(fig, OUTPUT_FILENAME, formats=EXPORT_FORMATS, dpi=DPI)

---
## 🎯 Exhibit 14: Scenario Analysis Valuation Comparison & Multi-Year Net Profit Trajectory
* **Panel A**: Valuation outputs (`Intrinsic @ Dec-25`, `12M Pure Intrinsic`, `12M Blended Target Price`) across Bear (`29.17%`), Base (`50.00%`), Bull (`20.83%`), and Probability-Weighted (`100%`) cases.
* **Panel B**: Projected Net Profit (`PATMI`, `IDR tn`) in `FY26E`, `FY27E`, `FY30E`, and `FY35E` across all scenarios.

In [ ]:
# ==============================================================================
# 🎨 USER CUSTOMIZATION BLOCK — EXHIBIT 14 (EDIT SIZE, LAYOUT, COLORS & FONTS)
# ==============================================================================
LAYOUT_MODE      = "horizontal"         # "horizontal" (1x2), "vertical" (2x1), "panel_a_only", "panel_b_only"
FIG_SIZE         = (11.0, 4.0)
DPI              = 300
EXPORT_FORMATS   = ["png"]
OUTPUT_FILENAME  = "exhibit14_scenario_analysis_matrix"

# Colors
COLOR_DEC25      = "#005F5B"            # Intrinsic @ Dec-25 (Dark Teal)
COLOR_INTR_12M   = "#00A39D"            # 12M Pure Intrinsic (BSI Teal)
COLOR_BLEND_12M  = "#F37021"            # 12M Blended Target Price (BSI Orange)
COLOR_NI_26      = "#85C1E9"            # FY26E PATMI bar (Sky Blue)
COLOR_SPOT_LINE  = "#C0392B"            # Current Price line (Crimson)

# Typography
TITLE_SIZE       = 9.2
AXIS_LABEL_SIZE  = 8.5
TICK_SIZE        = 7.6
ANNOT_SIZE       = 7.0
LEGEND_SIZE      = 6.9

# ==============================================================================
# 📊 CHART GENERATION LOGIC
# ==============================================================================
sc = data["scenarios"]
cases = ["Bear Case\n(29.17% Prob)", "Base Case\n(50.00% Prob)", "Bull Case\n(20.83% Prob)", "Prob-Weighted\n(100.0% Total)"]
x = np.arange(len(cases))

fig, ax1, ax2, show_a, show_b = create_panel_canvas(LAYOUT_MODE, FIG_SIZE)

if show_a:
    v_dec = sc["ddm_dec25"]
    v_12m = sc["ddm_12m"]
    v_bld = sc["blended_12m"]

    ax1.bar(x - 0.24, v_dec, 0.23, color=COLOR_DEC25,     label="Intrinsic Value @ Dec-25 (IDR)")
    ax1.bar(x,        v_12m, 0.23, color=COLOR_INTR_12M,  label="12M Pure Intrinsic DDM=RI (IDR)")
    ax1.bar(x + 0.24, v_bld, 0.23, color=COLOR_BLEND_12M, label="12M Blended Target Price (IDR)")
    for i in range(len(cases)):
        ax1.text(i + 0.24, v_bld[i] + 45, f"Rp{v_bld[i]:,.0f}\n({sc['upside'][i]*100:+.1f}%)",
                 ha="center", fontsize=ANNOT_SIZE, fontweight="bold", color=COLOR_BLEND_12M)

    ax1.axhline(1095.0, color=COLOR_SPOT_LINE, ls="--", lw=1.3, label="Current Price: IDR 1,095")
    ax1.set_xticks(x)
    ax1.set_xticklabels(cases, fontsize=TICK_SIZE, fontweight="bold")
    ax1.set_ylim(0, 3600)
    ax1.set_ylabel("Share Price / Valuation (IDR)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax1.set_title("A. Scenario Valuation Outputs (7 Downside : 5 Upside Evidence Weights)", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_DEC25, loc="left")
    ax1.legend(loc="upper left", fontsize=LEGEND_SIZE, frameon=True)

if show_b:
    ni26 = [v/1000 for v in sc["ni_fy26"]]
    ni27 = [v/1000 for v in sc["ni_fy27"]]
    ni30 = [v/1000 for v in sc["ni_fy30"]]
    ni35 = [v/1000 for v in sc["ni_fy35"]]

    ax2.bar(x - 0.27, ni26, 0.18, color=COLOR_NI_26,     label="FY26E Net Profit (IDR tn)")
    ax2.bar(x - 0.09, ni27, 0.18, color=COLOR_INTR_12M,  label="FY27E Net Profit (IDR tn)")
    ax2.bar(x + 0.09, ni30, 0.18, color=COLOR_DEC25,     label="FY30E Net Profit (IDR tn)")
    ax2.bar(x + 0.27, ni35, 0.18, color=COLOR_BLEND_12M, label="FY35E Net Profit (IDR tn)")
    for i in range(len(cases)):
        ax2.text(i + 0.27, ni35[i] + 0.6, f"Rp{ni35[i]:.1f}tn", ha="center", fontsize=ANNOT_SIZE, fontweight="bold", color=COLOR_BLEND_12M)

    ax2.set_xticks(x)
    ax2.set_xticklabels(cases, fontsize=TICK_SIZE, fontweight="bold")
    ax2.set_ylim(0, 36)
    ax2.set_ylabel("Net Profit / PATMI (IDR Trillion)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax2.set_title("B. Earnings Trajectory Across Bear, Base, Bull & Prob-Wtd Cases", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_DEC25, loc="left")
    ax2.legend(loc="upper left", fontsize=LEGEND_SIZE, frameon=True)

export_and_display(fig, OUTPUT_FILENAME, formats=EXPORT_FORMATS, dpi=DPI)

---
## ⚠️ Exhibit 15: 2D Risk Matrix Heatmap & One-at-a-Time Tornado Sensitivity Chart
* **Panel A**: 2D Risk Heatmap (`Likelihood × Valuation Impact`) mapping `MR1`, `MR2`, `IR1`, `IR2`, `BR1`, `BR2`, and `BR3`.
* **Panel B**: One-at-a-Time Tornado Sensitivity on the `IDR 1,881` 12-Month Blended Target Price across 8 core operational and macro drivers.

In [ ]:
# ==============================================================================
# 🎨 USER CUSTOMIZATION BLOCK — EXHIBIT 15 (EDIT SIZE, LAYOUT, COLORS & FONTS)
# ==============================================================================
LAYOUT_MODE      = "horizontal"         # "horizontal" (1x2), "vertical" (2x1), "panel_a_only", "panel_b_only"
FIG_SIZE         = (11.4, 4.2)
DPI              = 300
EXPORT_FORMATS   = ["png"]
OUTPUT_FILENAME  = "exhibit15_risk_matrix_and_tornado"

# Colors
RISK_HEAT_CMAP   = "YlOrRd"             # Background heatmap colormap
COLOR_DOWNSIDE   = "#F37021"            # Downside Tornado Bar (BSI Orange)
COLOR_UPSIDE     = "#00A39D"            # Upside Tornado Bar (BSI Teal)
COLOR_MACRO_RISK = "#C0392B"            # Macro Risk Badge (Crimson)
COLOR_IND_RISK   = "#F37021"            # Industry Risk Badge (BSI Orange)
COLOR_BUS_RISK   = "#005F5B"            # Business Risk Badge (Dark Teal)

# Typography
TITLE_SIZE       = 9.2
AXIS_LABEL_SIZE  = 8.2
TICK_SIZE        = 7.6
ANNOT_SIZE       = 6.6
LEGEND_SIZE      = 7.2

# ==============================================================================
# 📊 CHART GENERATION LOGIC
# ==============================================================================
fig, ax1, ax2, show_a, show_b = create_panel_canvas(LAYOUT_MODE, FIG_SIZE, width_ratios=[0.98, 1.22])

if show_a:
    gx, gy = np.meshgrid(np.linspace(0.5, 5.5, 100), np.linspace(0.5, 5.5, 100))
    ax1.imshow(gx * gy, extent=[0.5, 5.5, 0.5, 5.5], origin="lower", cmap=RISK_HEAT_CMAP, alpha=0.36, aspect="auto")

    risks = [
        ("MR1", "MR1: Higher-for-Longer BI Rate (5.75%)\n& Deposit Cost Stickiness", 4.2, 4.65, COLOR_MACRO_RISK, "left"),
        ("MR2", "MR2: MSCI Standalone Consultation\n& Foreign Outflow De-Rating",    3.6, 4.12, COLOR_MACRO_RISK, "left"),
        ("BR3", "BR3: Related-Party Musyarakah (+45%)\n& NPF Normalization",         2.0, 3.60, COLOR_BUS_RISK,   "right"),
        ("IR1", "IR1: UUS Spin-Offs (BTN/CIMB Syariah)\nCorporate Yield Compression",3.8, 3.08, COLOR_IND_RISK,   "left"),
        ("BR2", "BR2: Q1-27 Rights Issue Dilution\n& Deployment Lag",                2.1, 2.56, COLOR_BUS_RISK,   "right"),
        ("BR1", "BR1: IT/BYOND Opex Overrun\n(CIR Stays >54.5%)",                    3.4, 2.04, COLOR_BUS_RISK,   "left"),
        ("IR2", "IR2: Gold Price Volatility Impacting\nBullion/Rahn Fee Momentum",   2.2, 1.52, COLOR_IND_RISK,   "right"),
    ]
    for code, desc, lk, imp, col, side in risks:
        ax1.scatter(lk, imp, s=185, color=col, edgecolors="white", lw=1.5, zorder=5)
        ax1.text(lk, imp, code, ha="center", va="center", fontsize=ANNOT_SIZE + 0.4, fontweight="bold", color="white", zorder=6)
        dx = -12 if side == "left" else 12
        ha_align = "right" if side == "left" else "left"
        ax1.annotate(desc, xy=(lk, imp), xytext=(dx, 0), textcoords="offset points",
                     ha=ha_align, va="center", fontsize=ANNOT_SIZE, fontweight="bold", color="#1B2631",
                     bbox=dict(boxstyle="round,pad=0.16", fc="white", ec=col, alpha=0.94, lw=0.7))

    ax1.set_xlim(0.7, 5.0)
    ax1.set_ylim(1.15, 5.05)
    ax1.set_xticks([1.5, 3.0, 4.5])
    ax1.set_xticklabels(["Low", "Moderate", "High"], fontsize=TICK_SIZE, fontweight="bold")
    ax1.set_yticks([2.0, 3.25, 4.5])
    ax1.set_yticklabels(["Low", "Moderate", "High"], fontsize=TICK_SIZE, fontweight="bold")
    ax1.set_xlabel("Likelihood of Occurrence (7 Downside : 5 Upside Tally)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax1.set_ylabel("Valuation & Earnings Impact", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax1.set_title("A. 2D Risk Matrix: Macro (MR), Industry (IR) & Business (BR)", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_BUS_RISK, loc="left")

if show_b:
    tornado = sorted(data["tornado"], key=lambda t: abs(t["delta_high"]) + abs(t["delta_low"]))
    t_labels = [t["driver"] for t in tornado]
    base_pt  = tornado[0]["base_pt"]
    ty = np.arange(len(tornado))

    for i, t in enumerate(tornado):
        lo_pt = min(t["pt_low_shock"], t["pt_high_shock"])
        hi_pt = max(t["pt_low_shock"], t["pt_high_shock"])
        ax2.barh(i, lo_pt - base_pt, left=base_pt, height=0.55, color=COLOR_DOWNSIDE, alpha=0.88,
                 label="Downside Driver Shock" if i == 0 else "")
        ax2.barh(i, hi_pt - base_pt, left=base_pt, height=0.55, color=COLOR_UPSIDE, alpha=0.88,
                 label="Upside Driver Shock" if i == 0 else "")
        ax2.text(lo_pt - 8, i, f"Rp{lo_pt:,.0f} ({lo_pt-base_pt:+.0f})", ha="right", va="center",
                 fontsize=ANNOT_SIZE + 0.2, fontweight="bold", color=COLOR_MACRO_RISK)
        ax2.text(hi_pt + 8, i, f"Rp{hi_pt:,.0f} ({hi_pt-base_pt:+.0f})", ha="left", va="center",
                 fontsize=ANNOT_SIZE + 0.2, fontweight="bold", color=COLOR_BUS_RISK)

    ax2.axvline(base_pt, color="#1B2631", lw=1.6, label=f"Base 12M TP: IDR {base_pt:,.0f}")
    ax2.set_yticks(ty)
    ax2.set_yticklabels(t_labels, fontsize=TICK_SIZE, fontweight="bold")
    ax2.set_xlim(1500, 2260)
    ax2.set_xlabel("12-Month Blended Target Price (IDR/share)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax2.set_title("B. Tornado Sensitivity on 12M Blended Target Price (Base = IDR 1,881)", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_BUS_RISK, loc="left")
    ax2.legend(loc="lower right", fontsize=LEGEND_SIZE, frameon=True)

export_and_display(fig, OUTPUT_FILENAME, formats=EXPORT_FORMATS, dpi=DPI)

---
## 🌱 Exhibit 16: Customized Islamic Banking ESG Scorecard (`91.1 / 100`) & Valuation Transmission Link
* **Panel A**: BSI vs. KBMI-4 Peer Average across Environmental (`88.5`), Social (`94.0`), Governance (`91.2`), Cyber Resilience (`86.0`), and Composite Score (`91.1 vs. 82.4`).
* **Panel B**: Quantitative transmission of ESG/Governance into Cost of Equity (`0 bps` governance penalty in `Ke = 11.23% -> 12.25%`), Scenario Weights (`29.2%` Bear), and Terminal ROE (`14.25%`).

In [ ]:
# ==============================================================================
# 🎨 USER CUSTOMIZATION BLOCK — EXHIBIT 16 (EDIT SIZE, LAYOUT, COLORS & FONTS)
# ==============================================================================
LAYOUT_MODE      = "horizontal"         # "horizontal" (1x2), "vertical" (2x1), "panel_a_only", "panel_b_only"
FIG_SIZE         = (11.2, 3.9)
DPI              = 300
EXPORT_FORMATS   = ["png"]
OUTPUT_FILENAME  = "exhibit16_esg_scorecard_and_ke_link"

# Colors
COLOR_BSI_ESG    = "#00A39D"            # BSI ESG Score Bar (BSI Teal)
COLOR_PEER_ESG   = "#95A5A6"            # Peer Average ESG Bar (Muted Gray)
COLOR_DARK       = "#005F5B"            # Dark Teal Titles & Text
COLOR_ORANGE     = "#F37021"            # Governance Watch Bar (BSI Orange)
COLOR_GREEN      = "#1E8449"            # Sharia Moat Bar (Emerald Green)
COLOR_GOLD       = "#D4AF37"            # CRP Bar (Sharia Gold)

# Typography
TITLE_SIZE       = 9.2
AXIS_LABEL_SIZE  = 8.2
TICK_SIZE        = 7.4
ANNOT_SIZE       = 7.2
LEGEND_SIZE      = 7.5

# ==============================================================================
# 📊 CHART GENERATION LOGIC
# ==============================================================================
fig, ax1, ax2, show_a, show_b = create_panel_canvas(LAYOUT_MODE, FIG_SIZE, width_ratios=[1.1, 1.05])

if show_a:
    pillars = [
        "Environmental: Green Sukuk &\nSustainable Financing (Rp 66.5tn)",
        "Social: MSME/KUR Inclusion,\nZakat (Rp 316bn) & Gender Diversity",
        "Governance: Danantara Persero\nBoard, DSN-MUI & G20 Rights",
        "Digital & Cyber Resilience:\nBYOND Super-App & ISO 27001",
        "COMPOSITE SHARIA ESG SCORE\n(Weighted 25% E / 35% S / 40% G)"
    ]
    bris_s = [88.5, 94.0, 91.2, 86.0, 91.1]
    peer_s = [79.0, 81.5, 84.0, 85.0, 82.4]
    y = np.arange(len(pillars))[::-1]

    ax1.barh(y + 0.17, bris_s, height=0.33, color=COLOR_BSI_ESG,  label="BSI (BRIS.JK)")
    ax1.barh(y - 0.17, peer_s, height=0.33, color=COLOR_PEER_ESG, label="KBMI-4 Peer Average")
    for yp, bs, ps in zip(y, bris_s, peer_s):
        ax1.text(bs + 0.8, yp + 0.17, f"{bs:.1f}", va="center", fontsize=ANNOT_SIZE + 0.3, fontweight="bold", color=COLOR_DARK)
        ax1.text(ps + 0.8, yp - 0.17, f"{ps:.1f}", va="center", fontsize=ANNOT_SIZE, color="#5D6D7E")

    ax1.set_yticks(y)
    ax1.set_yticklabels(pillars, fontsize=TICK_SIZE, fontweight="bold")
    ax1.set_xlim(0, 106)
    ax1.set_xlabel("Customized Islamic Banking ESG Score (0–100 Scale)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax1.set_title("A. Customized Islamic Banking ESG & G20 Governance Scorecard", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_DARK, loc="left")
    ax1.legend(loc="lower right", fontsize=LEGEND_SIZE, frameon=True)

if show_b:
    trans_items = [
        "Base IDR Risk-Free Rate (7.18% - 1.62%)",
        "+ Full Country Risk Premium (λ = 1.00)",
        "+ β (0.7588→1.00) × Mature ERP (4.23%)",
        "+ ESG / Size / Specific Risk Premium",
        "= Year-1 Ke (11.23%) → Stable Ke (12.25%)",
        "Governance Watch: Related-Party Musyarakah\n(+45% YTD) & Cyber Opex → Bear Weight 29.2%",
        "Sharia Moat & Social Inclusion (+200bp Spread)\n→ Terminal ROE 14.25% (Base) / 16.25% (Bull)"
    ]
    trans_vals = [5.56, 2.46, 3.21, 0.00, 11.23, 12.25, 14.25]
    t_cols = [COLOR_BSI_ESG, COLOR_GOLD, COLOR_BSI_ESG, COLOR_GREEN, COLOR_DARK, COLOR_ORANGE, COLOR_GREEN]
    ty = np.arange(len(trans_items))[::-1]
    ax2.barh(ty, trans_vals, height=0.52, color=t_cols, alpha=0.88)
    annots = [
        "5.56% (10Y IndoGB net of Baa2 spread)",
        "+2.46% (Damodaran Jan-26 CRP)",
        "+3.21% Y1 (+4.23% by FY35E)",
        "0.00% (No ESG/Governance Penalty)",
        "11.23% Y1 / 12.25% Terminal Ke",
        "12.25% Bear ROE_T (29.17% Prob)",
        "14.25% Base ROE_T (50.0% Prob)"
    ]
    for yp, v, an in zip(ty, trans_vals, annots):
        ax2.text(v + 0.25, yp, an, va="center", fontsize=ANNOT_SIZE - 0.1, fontweight="bold", color="#1B2631")

    ax2.set_yticks(ty)
    ax2.set_yticklabels(trans_items, fontsize=TICK_SIZE - 0.2, fontweight="bold")
    ax2.set_xlim(0, 24.5)
    ax2.set_xlabel("Rate / Cost of Equity / Terminal ROE Impact (%)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax2.set_title("B. ESG-to-Valuation Transmission: Ke Build & Scenario Weighting", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_DARK, loc="left")

export_and_display(fig, OUTPUT_FILENAME, formats=EXPORT_FORMATS, dpi=DPI)

---
## 📉 Exhibit 17: 5-Year Historical Trailing `P/B` Band (`Feb 2021 – Oct 2026`) & 4-Specification Beta Audit
* **Panel A**: `BRIS.JK` trailing `P/B` multiple vs. 5-year historical mean (`2.48x`) and `±1SD / ±2SD` bands, showing current `0.99x` Jun-26 BV (`0.86x` FY26E BV) near `-2SD`.
* **Panel B**: Target Price sensitivity across all 4 Beta specifications (`Beta_Evidence` tab: TV 1Y Raw `0.64`, 60M Monthly `0.69`, KSI Broker `0.94`, and Manual Unadjusted `1.00`).

In [ ]:
# ==============================================================================
# 🎨 USER CUSTOMIZATION BLOCK — EXHIBIT 17 (EDIT SIZE, LAYOUT, COLORS & FONTS)
# ==============================================================================
LAYOUT_MODE      = "horizontal"         # "horizontal" (1x2), "vertical" (2x1), "panel_a_only", "panel_b_only"
FIG_SIZE         = (11.0, 4.0)
DPI              = 300
EXPORT_FORMATS   = ["png"]
OUTPUT_FILENAME  = "exhibit17_historical_pb_band_beta"

# Colors
COLOR_PB_LINE    = "#00A39D"            # Trailing P/B line & Base TP bars (BSI Teal)
COLOR_MEAN_LINE  = "#005F5B"            # 5Y Mean P/B line (Dark Teal)
COLOR_SD1_PLUS   = "#D4AF37"            # +1SD line (Sharia Gold)
COLOR_SD1_MINUS  = "#F37021"            # -1SD line & Prob-Wtd TP bars (BSI Orange)
COLOR_SD2_MINUS  = "#C0392B"            # -2SD line & Current Price callout (Crimson)

# Typography
TITLE_SIZE       = 9.2
AXIS_LABEL_SIZE  = 8.5
TICK_SIZE        = 7.1
ANNOT_SIZE       = 7.0
LEGEND_SIZE      = 6.9

# ==============================================================================
# 📊 CHART GENERATION LOGIC
# ==============================================================================
dates = data["prices_monthly"]["dates"]
prices = np.array(data["prices_monthly"]["bris"])
bvps_anchors = np.interp(np.arange(len(prices)), [0, 10, 22, 34, 46, 58, 68],
                         [580, 608.2, 726.4, 839.8, 976.4, 1126.2, 1106.5])
pb_series = np.append(prices / bvps_anchors, 0.9896)
dates_all = dates + ["16-Oct-26"]
x = np.arange(len(pb_series))
mean_pb = float(np.mean(pb_series))
sd_pb   = float(np.std(pb_series))

fig, ax1, ax2, show_a, show_b = create_panel_canvas(LAYOUT_MODE, FIG_SIZE)

if show_a:
    ax1.plot(x, pb_series, color=COLOR_PB_LINE, lw=2.2, label="BRIS Trailing P/B (x)")
    ax1.axhline(mean_pb, color=COLOR_MEAN_LINE, ls="-", lw=1.4, label=f"5Y Mean P/B: {mean_pb:.2f}x")
    ax1.axhline(mean_pb + sd_pb, color=COLOR_SD1_PLUS,  ls="--", lw=1.1, label=f"+1SD: {mean_pb+sd_pb:.2f}x")
    ax1.axhline(mean_pb - sd_pb, color=COLOR_SD1_MINUS, ls="--", lw=1.1, label=f"-1SD: {mean_pb-sd_pb:.2f}x")
    ax1.axhline(mean_pb + 2*sd_pb, color="#5D6D7E",     ls=":",  lw=1.0, label=f"+2SD: {mean_pb+2*sd_pb:.2f}x")
    ax1.axhline(mean_pb - 2*sd_pb, color=COLOR_SD2_MINUS, ls=":", lw=1.2, label=f"-2SD: {mean_pb-2*sd_pb:.2f}x")
    ax1.scatter(len(pb_series)-1, 0.9896, color=COLOR_SD2_MINUS, s=50, zorder=6)
    ax1.annotate("16-Oct-26: 0.99x Jun-26 BV\n(0.86x FY26E BV — Near -2SD!)\nBase TP Implies 1.47x FY26E BV",
                 xy=(len(pb_series)-1, 0.99), xytext=(-150, 24), textcoords="offset points",
                 fontsize=ANNOT_SIZE + 0.2, fontweight="bold", color=COLOR_SD2_MINUS,
                 bbox=dict(boxstyle="round,pad=0.22", fc="white", ec=COLOR_SD2_MINUS, lw=0.9),
                 arrowprops=dict(arrowstyle="->", color=COLOR_SD2_MINUS, lw=1.0))

    tick_idx = list(range(0, len(dates_all) - 5, 11)) + [len(dates_all) - 1]
    ax1.set_xticks(tick_idx)
    ax1.set_xticklabels([dates_all[i][:7] if i < len(dates_all)-1 else "16-Oct-26" for i in tick_idx], fontsize=TICK_SIZE, rotation=16)
    ax1.set_ylim(0.5, 5.2)
    ax1.set_ylabel("Trailing Price-to-Book Multiple (x)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax1.set_title("A. 5-Year Historical P/B Band (Feb 2021 – Oct 2026)", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_MEAN_LINE, loc="left")
    ax1.legend(loc="upper right", fontsize=LEGEND_SIZE - 0.1, ncol=2, frameon=True)

if show_b:
    bspecs = data["beta_specs"]
    blbls = [
        "1. TV 1Y Raw\n(0.64 → 0.76 Blume)\n[Ke=11.23%]",
        "2. 60M Monthly\n(0.69 → 0.79 Blume)\n[Ke=11.38%]",
        "3. KSI Broker\n(0.94 → 0.96 Blume)\n[Ke=12.07%]",
        "4. Manual Unadj.\n(1.00 → 1.00 Blume)\n[Ke=12.25%]"
    ]
    xb = np.arange(len(bspecs))
    ax2.bar(xb - 0.18, [b["pt_base"] for b in bspecs], 0.35, color=COLOR_PB_LINE,   label="Base 12M Blended TP (IDR)")
    ax2.bar(xb + 0.18, [b["pt_prob"] for b in bspecs], 0.35, color=COLOR_SD1_MINUS, label="Prob-Wtd 12M TP (IDR)")
    for i, b in enumerate(bspecs):
        ax2.text(i - 0.18, b["pt_base"] + 25, f"Rp{b['pt_base']:,.0f}", ha="center", fontsize=ANNOT_SIZE, fontweight="bold", color=COLOR_MEAN_LINE)
        ax2.text(i + 0.18, b["pt_prob"] + 25, f"Rp{b['pt_prob']:,.0f}\n(+{b['upside_prob']*100:.1f}%)", ha="center", fontsize=ANNOT_SIZE - 0.1, fontweight="bold", color=COLOR_SD1_MINUS)

    ax2.axhline(1095.0, color=COLOR_SD2_MINUS, ls="--", lw=1.2, label="Current Price: IDR 1,095")
    ax2.set_xticks(xb)
    ax2.set_xticklabels(blbls, fontsize=TICK_SIZE, fontweight="bold")
    ax2.set_ylim(0, 2350)
    ax2.set_ylabel("12-Month Target Price (IDR/share)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax2.set_title("B. Beta Specification Audit: BUY Rating Holds Across All 4 Betas", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_MEAN_LINE, loc="left")
    ax2.legend(loc="lower right", fontsize=LEGEND_SIZE, frameon=True)

export_and_display(fig, OUTPUT_FILENAME, formats=EXPORT_FORMATS, dpi=DPI)

---
## ♟️ Exhibit 18: Porter's Five Forces 5-Level Concentric Circle Radar (`2.74 / 5.0`) & Relative Blend Sensitivity
* **Panel A**: 5-Level Concentric Circle Radar (`Level 1 = Very Low Threat / Strong Moat` to `Level 5 = Severe Threat`, with `Level 3 = Neutral Midpoint Circle`), mapping BSI's 5 competitive forces (`1.8` New Entrants, `2.2` Depositors, `3.1` Borrowers, `3.2` Rivalry, `3.4` Substitutes).
* **Panel B**: Target Price sensitivity to Relative Valuation blend weight from `0%` (`100% Intrinsic DDM=RI = IDR 2,206`) to `50%` (`IDR 1,718`).

In [ ]:
# ==============================================================================
# 🎨 USER CUSTOMIZATION BLOCK — EXHIBIT 18 (EDIT SIZE, LAYOUT, COLORS & FONTS)
# ==============================================================================
LAYOUT_MODE      = "horizontal"         # "horizontal" (1x2), "vertical" (2x1), "panel_a_only", "panel_b_only"
FIG_SIZE         = (11.2, 4.2)
DPI              = 300
EXPORT_FORMATS   = ["png"]
OUTPUT_FILENAME  = "exhibit18_porters_five_forces_radar"

# Radar Polygon & Ring Colors
COLOR_POLY_FILL  = "#00A39D"            # BSI Teal Polygon Fill
COLOR_POLY_EDGE  = "#005F5B"            # Dark Teal Polygon Border
RING_FILLS       = ["#FDEDEC", "#FEF5EC", "#F8F9F9", "#E6F5F4", "#D1F2EB"] # Outer Ring 5 -> Inner Ring 1
VERTEX_COLORS    = ["#1E8449", "#00A39D", "#D4AF37", "#F37021", "#F37021"] # New Entrants, Depositors, Borrowers, Rivalry, Substitutes

# Panel B Bar Colors
COLOR_TOD_BAR    = "#005F5B"            # Blended Value Today bar (Dark Teal)
COLOR_12M_BAR    = "#00A39D"            # Blended 12M Target Price bar (BSI Teal)
COLOR_HIGHLIGHT  = "#F37021"            # Base Case 33.3% text highlight (BSI Orange)
COLOR_SPOT_LINE  = "#C0392B"            # Current Price line (Crimson)

# Typography
TITLE_SIZE       = 9.1
AXIS_LABEL_SIZE  = 8.2
TICK_SIZE        = 7.1
ANNOT_SIZE       = 6.5
LEGEND_SIZE      = 7.0

# ==============================================================================
# 📊 CHART GENERATION LOGIC
# ==============================================================================
fig, ax1, ax2, show_a, show_b = create_panel_canvas(LAYOUT_MODE, FIG_SIZE, width_ratios=[1.12, 0.98])

if show_a:
    ring_specs = [
        (5.0, RING_FILLS[0], "#E6B0AA", "5"),
        (4.0, RING_FILLS[1], "#FAD7A0", "4"),
        (3.0, RING_FILLS[2], "#5D6D7E", "3 (Neutral)"),
        (2.0, RING_FILLS[3], "#A3E4D7", "2"),
        (1.0, RING_FILLS[4], "#76D7C4", "1"),
    ]
    for r_lvl, fc_col, ec_col, lbl_lvl in ring_specs:
        is_neut = (r_lvl == 3.0)
        circ = mpatches.Circle((0, 0), r_lvl, facecolor=fc_col, edgecolor=ec_col,
                               ls="--" if is_neut else "-", lw=1.4 if is_neut else 0.9, zorder=1)
        ax1.add_patch(circ)
        ax1.text(0.12, r_lvl - 0.18, lbl_lvl, fontsize=ANNOT_SIZE - 0.2, fontweight="bold",
                 color="#1B2631" if is_neut else "#5D6D7E", ha="left", va="top", zorder=4)

    angles_deg = [90, 18, -54, -126, 162]
    angles_rad = [np.deg2rad(a) for a in angles_deg]
    forces_meta = [
        ("Threat of New Entrants",  "1.8 / 5.0 (LOW)", "Rp 10tn BUS Cap & DSN-MUI",   1.8, VERTEX_COLORS[0], "center", "bottom", (0, 5.35)),
        ("Power of Depositors",     "2.2 / 5.0 (LOW)", "61.6% CASA & 7.69M Hajj",     2.2, VERTEX_COLORS[1], "left",   "center", (5.35*np.cos(np.deg2rad(18)),   5.35*np.sin(np.deg2rad(18)))),
        ("Power of Borrowers",      "3.1 / 5.0 (MOD)", "Corp Spreads vs Retail Gold", 3.1, VERTEX_COLORS[2], "left",   "top",    (5.35*np.cos(np.deg2rad(-54)),  5.35*np.sin(np.deg2rad(-54)))),
        ("Intra-Industry Rivalry",  "3.2 / 5.0 (MOD)", "BSN & CIMB UUS Spin-Offs",    3.2, VERTEX_COLORS[3], "right",  "top",    (5.35*np.cos(np.deg2rad(-126)), 5.35*np.sin(np.deg2rad(-126)))),
        ("Threat of Substitutes",   "3.4 / 5.0 (MOD)", "Conv. KBMI-4 & Fintechs",     3.4, VERTEX_COLORS[4], "right",  "center", (5.35*np.cos(np.deg2rad(162)),  5.35*np.sin(np.deg2rad(162)))),
    ]

    for theta in angles_rad:
        ax1.plot([0, 5.0 * np.cos(theta)], [0, 5.0 * np.sin(theta)], color="#BDC3C7", lw=0.9, ls=":", zorder=2)

    poly_pts = [(m[3] * np.cos(th), m[3] * np.sin(th)) for m, th in zip(forces_meta, angles_rad)]
    poly_patch = mpatches.Polygon(poly_pts, closed=True, facecolor=COLOR_POLY_FILL, alpha=0.30, edgecolor=COLOR_POLY_EDGE, lw=2.3, zorder=3)
    ax1.add_patch(poly_patch)

    for (title_f, score_str, sub_f, sc_val, v_col, ha_a, va_a, (lx, ly)), th in zip(forces_meta, angles_rad):
        vx, vy = sc_val * np.cos(th), sc_val * np.sin(th)
        ax1.scatter(vx, vy, s=75, color=v_col, edgecolors="white", lw=1.5, zorder=5)
        b_col = v_col if v_col != "#D4AF37" else "#9A7D0A"
        ax1.text(lx, ly, f"{title_f}: {score_str}\n({sub_f})", ha=ha_a, va=va_a,
                 fontsize=ANNOT_SIZE, fontweight="bold", color="#1B2631",
                 bbox=dict(boxstyle="round,pad=0.20", fc="white", ec=b_col, lw=0.9, alpha=0.95), zorder=6)

    ax1.text(0, -6.85, "Concentric Circles: 1 = Very Low Threat (Strong Moat)  |  3 = Neutral Benchmark (Dashed)  |  5 = Severe Threat",
             ha="center", va="center", fontsize=ANNOT_SIZE, fontweight="bold", color=COLOR_POLY_EDGE,
             bbox=dict(boxstyle="round,pad=0.20", fc="#E6F5F4", ec=COLOR_POLY_FILL, lw=0.7))

    ax1.set_xlim(-10.8, 10.8)
    ax1.set_ylim(-7.5, 7.5)
    ax1.set_aspect("equal")
    ax1.axis("off")
    ax1.set_title("A. Porter's Five Forces 5-Level Circle Radar (Composite: 2.74 / 5.0 Favorable)",
                  fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_POLY_EDGE, loc="left", pad=6)

if show_b:
    w_rel = ["0% Rel\n(100% DDM=RI)", "20% Rel\n(80% DDM=RI)", "25% Rel\n(75% DDM=RI)", "33.3% Rel\n[MODEL BASE]", "40% Rel\n(60% DDM=RI)", "50% Rel\n(50% DDM=RI)"]
    v_tod = [2019.84, 1844.32, 1800.45, 1727.32, 1668.81, 1581.05]
    v_12m = [2206.50, 2011.27, 1962.47, 1881.12, 1816.05, 1718.44]
    xw = np.arange(len(w_rel))

    ax2.bar(xw - 0.17, v_tod, 0.34, color=COLOR_TOD_BAR, label="Blended Value Today (IDR/sh)")
    ax2.bar(xw + 0.17, v_12m, 0.34, color=COLOR_12M_BAR, label="Blended 12M Target Price (IDR/sh)")
    for i, (vt, v12) in enumerate(zip(v_tod, v_12m)):
        ax2.text(i + 0.17, v12 + 30, f"Rp{v12:,.0f}", ha="center", fontsize=ANNOT_SIZE + 0.5, fontweight="bold",
                 color=COLOR_HIGHLIGHT if i == 3 else COLOR_TOD_BAR)

    ax2.axhline(1095.0, color=COLOR_SPOT_LINE, ls="--", lw=1.2, label="Current Price: IDR 1,095")
    ax2.set_xticks(xw)
    ax2.set_xticklabels(w_rel, fontsize=TICK_SIZE, fontweight="bold")
    ax2.set_ylim(0, 2550)
    ax2.set_ylabel("Implied Share Price (IDR)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax2.set_title("B. Target Price Sensitivity to Relative Valuation Weight (0% to 50%)", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_TOD_BAR, loc="left")
    ax2.legend(loc="lower right", fontsize=LEGEND_SIZE, frameon=True)

export_and_display(fig, OUTPUT_FILENAME, formats=EXPORT_FORMATS, dpi=DPI)

---
## 🎲 Bonus Exhibit 19: Correlated 10,000-Trial Monte Carlo Valuation Simulation *(Optional Add-On)*
* **What it shows**: A calibration-locked `10,000`-iteration Cholesky-correlated Monte Carlo simulation anchored directly on the exact Tornado sensitivities and Bear (`IDR 1,325.18`) / Base (`IDR 1,881.12`) / Bull (`IDR 2,532.11`) bounds of `BSI_ValuationReport_Model_Claud.xlsx`, preserving the `DDM = RI` clean-surplus identity.

In [ ]:
# ==============================================================================
# 🎨 USER CUSTOMIZATION BLOCK — BONUS EXHIBIT 19 (MONTE CARLO SIMULATION)
# ==============================================================================
LAYOUT_MODE      = "horizontal"         # "horizontal" (1x2), "vertical" (2x1), "panel_a_only", "panel_b_only"
FIG_SIZE         = (11.0, 4.0)
DPI              = 300
EXPORT_FORMATS   = ["png"]
OUTPUT_FILENAME  = "exhibit19_monte_carlo_simulation"

N_SIMULATIONS    = 10000                # Number of Monte Carlo draws
RANDOM_SEED      = 42                   # Reproducible random seed
NUM_BINS         = 65                   # Histogram bin count

# Colors
COLOR_HIST_BARS  = "#00A39D"            # Histogram distribution fill (BSI Teal)
COLOR_CDF_LINE   = "#005F5B"            # Cumulative probability curve (Dark Teal)
COLOR_BASE_LINE  = "#F37021"            # Base TP / Mean line (BSI Orange)
COLOR_BUY_THRESH = "#1E8449"            # BUY Rating Threshold >= IDR 1,259 (Emerald)
COLOR_SPOT_LINE  = "#C0392B"            # Current Price IDR 1,095 (Crimson)

# Typography
TITLE_SIZE       = 9.2
AXIS_LABEL_SIZE  = 8.4
TICK_SIZE        = 7.6
ANNOT_SIZE       = 7.1
LEGEND_SIZE      = 6.9

# ==============================================================================
# 📊 CORRELATED MONTE CARLO ENGINE (CHOLESKY DECOMPOSITION ON 8 MODEL DRIVERS)
# ==============================================================================
np.random.seed(RANDOM_SEED)

# 8 Core Drivers from Sheet1!B447:I454 Tornado Table:
# [0] Gross Yield, [1] Ke, [2] Fee Growth, [3] TD Rate, [4] Fin Growth, [5] CoC, [6] CIR, [7] Terminal g
# Correlation matrix capturing banking ALCO & macro coupling:
corr_matrix = np.array([
    [ 1.00,  0.25,  0.15,  0.55, -0.20,  0.20, -0.10,  0.10],  # Gross Yield
    [ 0.25,  1.00, -0.15,  0.45, -0.35,  0.30,  0.15, -0.20],  # Cost of Equity (Ke)
    [ 0.15, -0.15,  1.00, -0.10,  0.40, -0.15, -0.35,  0.30],  # Fee Growth
    [ 0.55,  0.45, -0.10,  1.00, -0.30,  0.25,  0.10, -0.15],  # Time Deposit Rate
    [-0.20, -0.35,  0.40, -0.30,  1.00, -0.25, -0.30,  0.45],  # Financing Growth
    [ 0.20,  0.30, -0.15,  0.25, -0.25,  1.00,  0.20, -0.20],  # Cost of Credit (CoC)
    [-0.10,  0.15, -0.35,  0.10, -0.30,  0.20,  1.00, -0.15],  # CIR Overhead
    [ 0.10, -0.20,  0.30, -0.15,  0.45, -0.20, -0.15,  1.00],  # Terminal Growth g
])

L_chol = np.linalg.cholesky(corr_matrix)
z_uncorr = np.random.normal(0.0, 1.0, size=(N_SIMULATIONS, 8))
z_corr   = z_uncorr @ L_chol.T

# Exact per-1SD sensitivities calibrated to Sheet1 Tornado & Bear/Base/Bull distribution (7:5 asymmetric skew)
# Slight downside asymmetry (-26.53 IDR shift from Base 1,881.12 to Prob-Wtd 1,854.60)
sens_vec = np.array([210.0, -132.0, 95.0, -92.0, 72.0, -42.0, -38.0, 28.0])
base_tp  = data["valuation"]["blended_12m_base"]  # IDR 1,881.12

raw_deltas = z_corr @ sens_vec
# Nonlinear convexity for Ke and g + exact empirical mean calibration to Prob-Wtd TP (IDR 1,854.60)
nonlin_adj = 14.5 * (z_corr[:, 1]**2 - 1.0) - 26.53
mc_prices  = np.clip(base_tp + raw_deltas + nonlin_adj, 850.0, 3250.0)

p10, p25, p50, p75, p90 = np.percentile(mc_prices, [10, 25, 50, 75, 90])
mc_mean = float(np.mean(mc_prices))
mc_sd   = float(np.std(mc_prices))
prob_above_spot = float(np.mean(mc_prices > 1095.0) * 100.0)
prob_buy_rating = float(np.mean(mc_prices >= 1259.25) * 100.0)

fig, ax1, ax2, show_a, show_b = create_panel_canvas(LAYOUT_MODE, FIG_SIZE)

if show_a:
    counts, bins, patches = ax1.hist(mc_prices, bins=NUM_BINS, density=True, color=COLOR_HIST_BARS, alpha=0.78, edgecolor="white", lw=0.4)
    for b_left, patch in zip(bins[:-1], patches):
        if b_left < 1095.0:
            patch.set_facecolor(COLOR_SPOT_LINE)
        elif b_left < 1259.25:
            patch.set_facecolor("#D4AF37")

    ax1.axvline(1095.0,  color=COLOR_SPOT_LINE,  ls="--", lw=1.6, label=f"Current Price: IDR 1,095")
    ax1.axvline(1259.25, color=COLOR_BUY_THRESH, ls=":",  lw=1.6, label=f"BUY Threshold (+15%): IDR 1,259 ({prob_buy_rating:.1f}% Prob)")
    ax1.axvline(mc_mean, color=COLOR_BASE_LINE,  ls="-",  lw=2.1, label=f"MC Mean: IDR {mc_mean:,.0f} (Median: IDR {p50:,.0f})")
    ax1.axvline(p10,     color="#5D6D7E",        ls="-.", lw=1.2, label=f"P10 – P90 Range: IDR {p10:,.0f} – IDR {p90:,.0f}")
    ax1.axvline(p90,     color="#5D6D7E",        ls="-.", lw=1.2)

    ax1.set_xlabel("Simulated 12-Month Target Price (IDR / share)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax1.set_ylabel("Probability Density", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax1.set_title(f"A. Monte Carlo Target Price Distribution (N = {N_SIMULATIONS:,} Correlated Trials)", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_CDF_LINE, loc="left")
    ax1.legend(loc="upper right", fontsize=LEGEND_SIZE, frameon=True)

if show_b:
    sorted_p = np.sort(mc_prices)
    cdf_y = np.arange(1, len(sorted_p) + 1) / len(sorted_p) * 100.0
    ax2.plot(sorted_p, cdf_y, color=COLOR_CDF_LINE, lw=2.3, label="Cumulative Probability (CDF %)")
    ax2.fill_between(sorted_p, 0, cdf_y, color=COLOR_HIST_BARS, alpha=0.12)

    for pct, val, col in [(10, p10, "#5D6D7E"), (25, p25, COLOR_HIST_BARS), (50, p50, COLOR_BASE_LINE), (75, p75, COLOR_HIST_BARS), (90, p90, "#1E8449")]:
        ax2.scatter(val, pct, color=col, s=45, zorder=5)
        ax2.annotate(f"P{pct}: Rp {val:,.0f}", xy=(val, pct), xytext=(10, -4), textcoords="offset points",
                     fontsize=ANNOT_SIZE, fontweight="bold", color=col)

    ax2.axvline(1259.25, color=COLOR_BUY_THRESH, ls=":", lw=1.5, label=f"{prob_buy_rating:.1f}% of Trials Rate BUY (≥Rp 1,259)")
    ax2.set_ylim(0, 105)
    ax2.set_xlabel("Simulated 12-Month Target Price (IDR / share)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax2.set_ylabel("Cumulative Probability (%)", fontsize=AXIS_LABEL_SIZE, fontweight="bold")
    ax2.set_title(f"B. Cumulative Distribution Function & Percentile Benchmarks (SD = Rp {mc_sd:,.0f})", fontsize=TITLE_SIZE, fontweight="bold", color=COLOR_CDF_LINE, loc="left")
    ax2.legend(loc="lower right", fontsize=LEGEND_SIZE, frameon=True)

export_and_display(fig, OUTPUT_FILENAME, formats=EXPORT_FORMATS, dpi=DPI)